# CS781 Capstone · Phase 6 · Abstractive Legal Summarization

In [ ]:
import os, sys, subprocess, json, glob, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "huggingface_hub", "rouge-score", "bert-score"], check=False)
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
os.makedirs(f"{WORK}/p6summ/phase1_models", exist_ok=True); os.makedirs(f"{WORK}/tests", exist_ok=True)
os.chdir(WORK); sys.path.insert(0, WORK)
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE (enable GPU!)",
      "| CPUs:", os.cpu_count())

## Pipeline code (written to `p6summ/`; nothing to edit)

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/arch_a.py
"""Architecture A: decoder-only LLaMA-style (GQA, RoPE, SwiGLU).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (decoder-only-fine-tuned.ipynb / decoder-only-transformer-llama-style.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class GQAttention(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0 and n_head % n_kv_head == 0
        self.n_head, self.n_kv, self.hd = n_head, n_kv_head, d_model // n_head
        self.wq = nn.Linear(d_model, n_head * self.hd, bias=bias)
        self.wkv = nn.Linear(d_model, 2 * n_kv_head * self.hd, bias=bias)
        self.proj = nn.Linear(n_head * self.hd, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, pos0=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        k, v = self.wkv(x).view(B, T, 2, self.n_kv, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        if self.n_kv != self.n_head:
            rep = self.n_head // self.n_kv
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, -1)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_head, n_kv_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, pos0, cache)
        return x + self.ffn(self.norm2(x))

class LlamaLM(nn.Module):
    def __init__(self, vocab_size, block_size, pad_id=None, n_layer=12, n_head=10, n_kv_head=2, d_model=640,
                 ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[LlamaLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.pad_id, self.n_layer = block_size, pad_id, n_layer
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, block_size, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, n_kv_head, hidden, dropout, bias) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, pos0=0, caches=None, last_only=False):
        B, T = idx.shape
        assert pos0 + T <= self.block_size, f"position {pos0 + T} > block_size {self.block_size}"
        x = self.drop(self.tok_emb(idx))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = LlamaLM

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/arch_b.py
"""Architecture B: encoder-decoder.

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (B/notebookc976077ad3.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.hd = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, pos0=0, key_mask=None, causal=False, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        mask = None if key_mask is None else key_mask[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and T > 1 and mask is None,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        self.n_head, self.hd = n_head, d_model // n_head
        self.q = nn.Linear(d_model, d_model, bias=bias)
        self.kv = nn.Linear(d_model, 2 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, memory, mem_mask, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        if cache is not None and "ck" in cache:
            k, v = cache["ck"], cache["cv"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["ck"], cache["cv"] = k, v
        mask = None if mem_mask is None else mem_mask[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = SelfAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, key_mask):
        x = x + self.attn(self.norm1(x), rope, 0, key_mask=key_mask, causal=False)
        return x + self.ffn(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = RMSNorm(d_model), RMSNorm(d_model), RMSNorm(d_model)
        self.self_attn = SelfAttention(d_model, n_head, dropout, bias)
        self.cross_attn = CrossAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, memory, mem_mask, pos0=0, cache=None):
        x = x + self.self_attn(self.norm1(x), rope, pos0, causal=True, cache=cache)
        x = x + self.cross_attn(self.norm2(x), memory, mem_mask, cache=cache)
        return x + self.ffn(self.norm3(x))

class EncDecLM(nn.Module):
    def __init__(self, vocab_size, enc_block, dec_block, pad_id=None, n_enc_layer=6, n_dec_layer=6,
                 n_head=9, d_model=576, ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[EncDecLM] ignoring unknown config keys: {sorted(kw)}")
        self.enc_block, self.dec_block, self.pad_id = enc_block, dec_block, pad_id
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        hd = d_model // n_head
        self.enc_rope = RotaryEmbedding(hd, enc_block, rope_theta)
        self.dec_rope = RotaryEmbedding(hd, dec_block, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.enc_blocks = nn.ModuleList([EncoderBlock(d_model, n_head, hidden, dropout, bias) for _ in range(n_enc_layer)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(d_model, n_head, hidden, dropout, bias) for _ in range(n_dec_layer)])
        self.enc_norm, self.dec_norm = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_enc_layer + n_dec_layer)
        self.lm_head.weight = self.tok_emb.weight

    def encode(self, src, src_mask=None):
        assert src.size(1) <= self.enc_block, f"source length {src.size(1)} > enc_block {self.enc_block}"
        if src_mask is not None and bool(src_mask.all()):
            src_mask = None
        x = self.drop(self.tok_emb(src))
        for blk in self.enc_blocks:
            x = blk(x, self.enc_rope, src_mask)
        return self.enc_norm(x)

    def decode(self, dec_in, memory, mem_mask=None, pos0=0, caches=None, last_only=False):
        assert pos0 + dec_in.size(1) <= self.dec_block, f"decoder position {pos0 + dec_in.size(1)} > dec_block"
        if mem_mask is not None and bool(mem_mask.all()):
            mem_mask = None
        x = self.drop(self.tok_emb(dec_in))
        for i, blk in enumerate(self.dec_blocks):
            x = blk(x, self.dec_rope, memory, mem_mask, pos0, None if caches is None else caches[i])
        x = self.dec_norm(x)
        if last_only:
            x = x[:, -1:]
        return self.lm_head(x)

    def forward(self, src, src_mask, dec_in, targets=None):
        memory = self.encode(src, src_mask)
        logits = self.decode(dec_in, memory, src_mask)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = EncDecLM

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/arch_c.py
"""Architecture C: encoder/hybrid prefix-LM + MLM (UniLM-style).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (C/encoder-hybrid.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class HybridSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.hd = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, causal=True, pos0=0, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                           is_causal=causal and attn_mask is None and T > 1,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = HybridSelfAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, attn_mask, causal, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, attn_mask, causal, pos0, cache)
        return x + self.ffn(self.norm2(x))

class HybridLM(nn.Module):
    def __init__(self, vocab_size, block_size, pad_id=None, n_layer=10, n_head=10, d_model=640, ffn_mult=8 / 3,
                 dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, n_segments=3, **kw):
        super().__init__()
        if kw:
            print(f"[HybridLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.pad_id, self.n_layer = block_size, pad_id, n_layer
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.seg_emb = nn.Embedding(n_segments, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, block_size, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, hidden, dropout, bias) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        nn.init.zeros_(self.seg_emb.weight)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    @staticmethod
    def hybrid_mask(prefix_len, T):
        pos = torch.arange(T, device=prefix_len.device)
        in_prefix = pos[None, None, :] < prefix_len[:, None, None]
        causal = pos[None, :, None] >= pos[None, None, :]
        return (in_prefix | causal)[:, None]

    def forward(self, idx, prefix_len=None, seg=None, targets=None, pos0=0, caches=None, last_only=False):
        B, T = idx.shape
        assert pos0 + T <= self.block_size, f"position {pos0 + T} > block_size {self.block_size}"
        if prefix_len is None:
            prefix_len = torch.zeros(B, dtype=torch.long, device=idx.device)
        prefix_len = prefix_len.to(idx.device).long().clamp(0, T)
        if bool((prefix_len == 0).all()):
            attn_mask, causal = None, True
        elif bool((prefix_len == T).all()):
            attn_mask, causal = None, False
        else:
            attn_mask, causal = self.hybrid_mask(prefix_len, T), False
        if seg is None:
            seg = (torch.arange(T, device=idx.device)[None, :] >= prefix_len[:, None]).long()
        x = self.drop(self.tok_emb(idx) + self.seg_emb(seg))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, attn_mask, causal, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = HybridLM

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/arch_d.py
"""Architecture D: efficient decoder, sliding-window attention + MQA.

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (D/notebook7e00370b82.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class LocalMQAttention(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, window, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0 and n_head % n_kv_head == 0
        self.n_head, self.n_kv, self.hd, self.window = n_head, n_kv_head, d_model // n_head, window
        self.wq = nn.Linear(d_model, n_head * self.hd, bias=bias)
        self.wkv = nn.Linear(d_model, 2 * n_kv_head * self.hd, bias=bias)
        self.proj = nn.Linear(n_head * self.hd, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)
        self._mask_cache = {}

    def _block_mask(self, nb, device):
        key = (nb, str(device))
        m = self._mask_cache.get(key)
        if m is None:
            W = self.window
            a = torch.arange(W, device=device)[:, None]
            b = torch.arange(2 * W, device=device)[None, :]
            keep = ((b > a) & (b <= a + W))[None].repeat(nb, 1, 1)
            keep[0, :, :W] = False
            m = keep[None, None].expand(1, self.n_head, nb, W, 2 * W).reshape(1, self.n_head * nb, W, 2 * W)
            self._mask_cache = {key: m}
        return m

    def _blocked(self, q, k, v):
        B, H, T, hd = q.shape
        W, Hkv = self.window, k.size(1)
        Tp = -(-T // W) * W
        if Tp != T:
            q, k, v = (F.pad(t, (0, 0, 0, Tp - T)) for t in (q, k, v))
        nb = Tp // W
        k = F.pad(k, (0, 0, W, 0)).unfold(2, 2 * W, W).transpose(-1, -2)
        v = F.pad(v, (0, 0, W, 0)).unfold(2, 2 * W, W).transpose(-1, -2)
        rep = H // Hkv
        k = k[:, :, None].expand(B, Hkv, rep, nb, 2 * W, hd).reshape(B, H * nb, 2 * W, hd)
        v = v[:, :, None].expand(B, Hkv, rep, nb, 2 * W, hd).reshape(B, H * nb, 2 * W, hd)
        qb = q.reshape(B, H * nb, W, hd)
        y = F.scaled_dot_product_attention(qb, k, v, attn_mask=self._block_mask(nb, q.device),
                                           dropout_p=self.dropout if self.training else 0.0)
        return y.reshape(B, H, Tp, hd)[:, :, :T]

    def forward(self, x, rope, pos0=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        k, v = self.wkv(x).view(B, T, 2, self.n_kv, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None and "k" in cache:
            assert T == 1, "cached decoding feeds one token at a time"
            k = torch.cat([cache["k"], k], 2)[:, :, -self.window:]
            v = torch.cat([cache["v"], v], 2)[:, :, -self.window:]
            cache["k"], cache["v"] = k, v
            rep = self.n_head // self.n_kv
            y = F.scaled_dot_product_attention(q, k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1))
        else:
            y = self._blocked(q, k, v)
            if cache is not None:
                cache["k"], cache["v"] = k[:, :, -self.window:], v[:, :, -self.window:]
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, -1)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, window, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = LocalMQAttention(d_model, n_head, n_kv_head, window, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, pos0, cache)
        return x + self.ffn(self.norm2(x))

class EfficientLM(nn.Module):
    def __init__(self, vocab_size, block_size, window=256, pad_id=None, n_layer=12, n_head=10, n_kv_head=1,
                 d_model=640, ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, rope_max_len=8192,
                 tie_weights=True, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[EfficientLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.window, self.pad_id, self.n_layer = block_size, window, pad_id, n_layer
        self.max_pos = max(rope_max_len, block_size)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, self.max_pos, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, n_kv_head, window, hidden, dropout, bias)
                                     for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, pos0=0, caches=None, last_only=False):
        x = self.drop(self.tok_emb(idx))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = EfficientLM

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/common.py
"""Shared building blocks (RMSNorm, RoPE, SwiGLU, init).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (identical in A/B/C/D notebooks, cell 19),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def _ffn_hidden(d_model, mult, multiple_of=64):
    h = int(mult * d_model)
    return multiple_of * ((h + multiple_of - 1) // multiple_of)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head_dim"
        inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        T = x.size(-2)
        assert pos0 + T <= self.cos.size(0), f"position {pos0 + T} > RoPE table {self.cos.size(0)}"
        cos, sin = self.cos[pos0:pos0 + T], self.sin[pos0:pos0 + T]
        x1, x2 = x.float().chunk(2, dim=-1)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden, dropout=0.0, bias=False):
        super().__init__()
        self.w12 = nn.Linear(d_model, 2 * hidden, bias=bias)
        self.proj = nn.Linear(hidden, d_model, bias=bias)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.drop(self.proj(F.silu(a) * b))

def init_weights_(model, n_layer):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

def lm_loss(logits, targets):
    return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)

In [ ]:
%%writefile {WORK}/p6summ/phase1_models/__init__.py
from .arch_a import LlamaLM
from .arch_b import EncDecLM
from .arch_c import HybridLM
from .arch_d import EfficientLM

MODEL_CLASSES = {c.__name__: c for c in (LlamaLM, EncDecLM, HybridLM, EfficientLM)}

In [ ]:
%%writefile {WORK}/p6summ/__init__.py
"""CS781 Phase 6: abstractive legal summarisation on the Phase-1 legal language models."""

In [ ]:
%%writefile {WORK}/p6summ/tokenizer.py
"""Byte-level BPE tokenizer from Phase 1 (load-only copy) + character offsets for NER.

Phase 1 trained the merges; here we only *load* them (same ids as pretraining) and add
`encode_with_offsets`, which returns for every token the character span it covers in the original text.

Offsets are computed per pre-token: the regex gives each pre-token's char span, BPE splits the pre-token's
UTF-8 bytes into tokens, and a byte->char table maps every token's byte range back to characters.
"""
import hashlib
import json
import os
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import regex as re_u

SPECIAL_TOKENS = ["<|endoftext|>", "<|pad|>", "<|bos|>", "<|sep|>", "<|mask|>"]

FINE_PUNCT_PATTERN = (r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}{1,3}| ?[^\s\p{L}\p{N}]|\s*[\r\n]+|\s+(?!\S)|\s+""")

def _merge_ids(ids, pair, new_id):
    out, i, n = [], 0, len(ids)
    a, b = pair
    while i < n:
        if i < n - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

@dataclass
class Encoded:
    ids: list
    starts: list
    ends: list

class BPETokenizer:
    def __init__(self, merges, pattern, specials=SPECIAL_TOKENS, fine_punct=False):
        self.base_pattern = pattern
        self.fine_punct = fine_punct
        self.pattern = FINE_PUNCT_PATTERN if fine_punct else pattern
        self.pat = re_u.compile(self.pattern)
        self.merges = [tuple(m) for m in merges]
        self.specials = list(specials)
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        base = 256 + len(self.merges)
        self.n_regular = base
        self.special_to_id = {s: base + i for i, s in enumerate(self.specials)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.vocab_size = base + len(self.specials)
        self.cache = {}
        self._oc, self._oc_path, self._oc_new = None, None, 0

    def fingerprint(self):
        h = hashlib.sha1(self.pattern.encode())
        h.update(json.dumps(self.merges).encode())
        return h.hexdigest()[:12]

    def enable_disk_cache(self, cache_dir):
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        self._oc_path = Path(cache_dir) / f"offsets_{self.fingerprint()}.pkl"
        self._oc = {}
        if self._oc_path.exists():
            try:
                self._oc = pickle.load(open(self._oc_path, "rb"))
            except Exception:
                self._oc = {}
        return self

    def save_disk_cache(self):
        if self._oc is None or not self._oc_new:
            return
        disk = {}
        if self._oc_path.exists():
            try:
                disk = pickle.load(open(self._oc_path, "rb"))
            except Exception:
                disk = {}
        disk.update(self._oc)
        tmp = self._oc_path.with_suffix(f".{os.getpid()}.tmp")
        pickle.dump(disk, open(tmp, "wb"), protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, self._oc_path)
        self._oc, self._oc_new = disk, 0

    @classmethod
    def load(cls, path, fine_punct=False):
        d = json.load(open(path, encoding="utf-8"))
        return cls(d["merges"], d["pattern"], d.get("specials", SPECIAL_TOKENS), fine_punct=fine_punct)

    @property
    def pad_id(self): return self.special_to_id["<|pad|>"]
    @property
    def bos_id(self): return self.special_to_id["<|bos|>"]
    @property
    def mask_id(self): return self.special_to_id["<|mask|>"]

    def _encode_word(self, s):
        hit = self.cache.get(s)
        if hit is not None:
            return hit
        ids = list(s.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best, best_r = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best, best_r = p, r
            if best is None:
                break
            ids = _merge_ids(ids, best, 256 + best_r)
        if len(self.cache) > 500_000:
            self.cache.clear()
        self.cache[s] = ids
        return ids

    def encode(self, text):
        out = []
        for w in self.pat.findall(text):
            out.extend(self._encode_word(w))
        return out

    def encode_with_offsets(self, text):
        if self._oc is not None:
            key = hashlib.sha1(text.encode("utf-8")).hexdigest()
            hit = self._oc.get(key)
            if hit is not None:
                return Encoded(hit[0].tolist(), hit[1].tolist(), hit[2].tolist())
            enc = self._encode_with_offsets(text)
            self._oc[key] = (np.asarray(enc.ids, dtype=np.int32), np.asarray(enc.starts, dtype=np.int32),
                             np.asarray(enc.ends, dtype=np.int32))
            self._oc_new += 1
            return enc
        return self._encode_with_offsets(text)

    def _encode_with_offsets(self, text):
        ids, starts, ends = [], [], []
        for m in self.pat.finditer(text):
            w, c0 = m.group(), m.start()
            wid = self._encode_word(w)
            if len(wid) == 1:
                ids.append(wid[0]); starts.append(c0); ends.append(m.end())
                continue
            b2c = []
            for ci, ch in enumerate(w):
                b2c.extend([ci] * len(ch.encode("utf-8")))
            b = 0
            for t in wid:
                nb = len(self.vocab[t])
                ids.append(t)
                starts.append(c0 + b2c[b])
                ends.append(c0 + b2c[b + nb - 1] + 1)
                b += nb
        return Encoded(ids, starts, ends)

    def decode(self, ids):
        return b"".join(self.vocab[int(i)] for i in ids if int(i) < self.n_regular).decode("utf-8", errors="replace")

In [ ]:
%%writefile {WORK}/p6summ/data.py
"""Phase-6 data: download, load, validate, held-out dev split, tokenise once (cached), statistics.

Layout (HF dataset Exploration-Lab/CS-781-Capstone, folder Phase_6_Summ):
    Train/train.json   [{"id": "doc_001", "document": "...", "summary": "Reference summary ..."}, ...]
    Test/test.json     [{"id": "doc_101", "document": "..."}, ...]
No public dev split: `dev_size` training documents are held out by a seeded hash of their id (the same split for
every run and every session), the rest is used for training.
The loader also accepts a dict {id: {...}}, JSON lines, a {"data": [...]} wrapper, the text under
"document"/"text"/"judgment"/..., the summary under "summary"/"abstract"/"headnote"/..., and paragraph/sentence lists
(joined), so a small format difference does not break it.

Documents are split into sentences and every sentence is tokenised separately (the space before a sentence stays
with it, so the concatenation is exactly the tokenisation of the whole document). Stored per document:
    tokens     uint16 array, the whole document
    sent_end   int32 array, token index where each sentence ends (cumulative)
    sent_score float32 array, salience of each sentence (TF-IDF centroid similarity, see `sentence_scores`)
Tokenising with the pure-Python Phase-1 BPE is the slowest CPU step, so it is done ONCE (multiprocessing) and
stored in cache_dir; every run (and both parallel GPU jobs) reuses it.
"""
import hashlib
import json
import math
import os
import pickle
import re
from collections import Counter
from pathlib import Path

import numpy as np

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_SUBDIR = "Phase_6_Summ"
ID_KEYS = ("id", "doc_id", "case_id", "uid", "name")
DOC_KEYS = ("document", "text", "doc", "judgment", "judgement", "source", "article", "case_text", "content", "body")
SUM_KEYS = ("summary", "abstract", "headnote", "headnotes", "reference", "target", "highlights", "summ")
SPLITS = ("train", "dev", "test")

def download(local_dir, token=None):
    from huggingface_hub import snapshot_download
    token = token or os.environ.get("HF_TOKEN")
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=local_dir, token=token,
                      allow_patterns=[f"{HF_SUBDIR}/*"])
    return Path(local_dir) / HF_SUBDIR

def find_data_dir(candidates=()):
    def ok(d):
        return (d / "Train" / "train.json").exists() and (d / "Test" / "test.json").exists()
    for c in candidates:
        if c and ok(Path(c)):
            return Path(c)
    for r in [Path(c) for c in candidates if c] + [Path("/kaggle/input")]:
        if not r.exists():
            continue
        for p in r.rglob("train.json"):
            d = p.parent.parent
            low = str(d).lower()
            if p.parent.name == "Train" and ok(d) and ("summ" in low or "phase_6" in low or "phase6" in low
                                                       or "phase-6" in low):
                return d
    return None

def file_path(data_dir, split):
    return Path(data_dir) / ("Test/test.json" if split == "test" else "Train/train.json")

def _read_json(path):
    txt = Path(path).read_text(encoding="utf-8")
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

def _records(raw):
    if isinstance(raw, dict):
        for k in ("data", "documents", "docs", "examples", "cases"):
            if isinstance(raw.get(k), list):
                return raw[k]
        recs = []
        for k, v in raw.items():
            if isinstance(v, dict):
                recs.append({"id": k, **v} if not any(x in v for x in ID_KEYS) else v)
            elif isinstance(v, str):
                recs.append({"id": k, "document": v})
            else:
                raise ValueError(f"unexpected value type for document {k}: {type(v)}")
        return recs
    if isinstance(raw, list):
        return raw
    raise ValueError(f"unexpected top-level JSON type {type(raw)}")

def _first(r, keys):
    for k in keys:
        if k in r and r[k] is not None:
            return k, r[k]
    return None, None

def normalise_text(t):
    if isinstance(t, list):
        t = " ".join("" if x is None else str(x) for x in t)
    return " ".join(str(t).split())

def norm_doc(r, where, need_summary):
    if not isinstance(r, dict):
        raise ValueError(f"{where}: record is {type(r)}, expected an object")
    _, rid = _first(r, ID_KEYS)
    _, doc = _first(r, DOC_KEYS)
    _, summ = _first(r, SUM_KEYS)
    if rid is None:
        raise ValueError(f"{where}: document without an id field: keys {sorted(r)}")
    if doc is None:
        raise ValueError(f"{where}: document {rid} has no text field: keys {sorted(r)}")
    if need_summary and summ is None:
        raise ValueError(f"{where}: document {rid} has no summary field: keys {sorted(r)}")
    return {"id": str(rid), "raw_id": rid, "document": normalise_text(doc),
            "summary": normalise_text(summ) if (need_summary and summ is not None) else None}

def load_file(data_dir, split_file, max_docs=None):
    p = file_path(data_dir, split_file)
    if not p.exists():
        raise FileNotFoundError(p)
    recs = _records(_read_json(p))
    if max_docs:
        recs = recs[:max_docs]
    docs = [norm_doc(r, f"{p.name}[{i}]", split_file != "test") for i, r in enumerate(recs)]
    seen, out, dup = set(), [], 0
    for d in docs:
        if d["id"] in seen:
            dup += 1
            continue
        seen.add(d["id"])
        out.append(d)
    if dup:
        print(f"[data] {p.name}: dropped {dup} repeated ids (first one kept)")
    return out

def dev_ids(ids, dev_size, seed=0):
    h = sorted(ids, key=lambda i: hashlib.sha1(f"{seed}:{i}".encode()).hexdigest())
    return set(h[:dev_size])

def describe_raw(data_dir, n_chars=300):
    for split in ("train", "test"):
        p = file_path(data_dir, split)
        raw = _read_json(p)
        recs = _records(raw)
        first = recs[0] if recs else {}
        short = {k: (v[:n_chars] + "...") if isinstance(v, str) and len(v) > n_chars else v for k, v in first.items()}
        print(f"{p.name}: {p.stat().st_size / 2**20:.1f} MB | top-level {type(raw).__name__} | {len(recs)} records | "
              f"keys {sorted(first)} | {json.dumps(short, default=str)[:2 * n_chars + 200]}")

_SENT_RE = re.compile(r"(?<=[.?!;:])\s+(?=[\"'(\[A-Z0-9])")
_WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
STOP = set("""a an the of to in and or for on at by with from as is are was were be been being this that these those it
its which who whom whose what when where why how not no nor but if then than so such there their them they he she his
her him we our us you your i me my mine shall will would should could can may might must do does did done has have had
having into upon under over about above below up down out off again further once all any both each few more most other
some same own only very s t also said per vs v""".split())

def split_sentences(text, max_chars=1500):
    if not text:
        return []
    cuts = [m.end() - 1 for m in _SENT_RE.finditer(text)]
    pieces, s = [], 0
    for c in cuts + [len(text)]:
        piece = text[s:c]
        while len(piece) > max_chars:
            k = piece.rfind(" ", 1, max_chars)
            k = k if k > 0 else max_chars
            pieces.append(piece[:k]); piece = piece[k:]
        if piece:
            pieces.append(piece)
        s = c
    return pieces

def content_words(text):
    return [w for w in _WORD_RE.findall(text.lower())
            if (w.isdigit() and len(w) >= 3) or (not w.isdigit() and w not in STOP)]

def sentence_scores(sentences):
    bags = [Counter(content_words(s)) for s in sentences]
    n = len(bags)
    if n == 0:
        return np.zeros(0, dtype=np.float32)
    df = Counter(w for b in bags for w in b)
    idf = {w: math.log(1 + n / c) for w, c in df.items()}
    cent = Counter()
    for b in bags:
        cent.update(b)
    cv = {w: c * idf[w] for w, c in cent.items()}
    cn = math.sqrt(sum(v * v for v in cv.values())) or 1.0
    out = np.zeros(n, dtype=np.float32)
    for i, b in enumerate(bags):
        if not b:
            continue
        sv = {w: c * idf[w] for w, c in b.items()}
        sn = math.sqrt(sum(v * v for v in sv.values())) or 1.0
        out[i] = sum(v * cv[w] for w, v in sv.items()) / (sn * cn)
    return out

_TOK = None

def _init_worker(tok_path):
    global _TOK
    from .tokenizer import BPETokenizer
    _TOK = BPETokenizer.load(tok_path)

def _encode_doc(item):
    doc, summ = item
    sents = split_sentences(doc) or ["."]
    toks, ends = [], []
    for s in sents:
        toks.extend(_TOK.encode(s))
        ends.append(len(toks))
    if not toks:
        toks, ends = _TOK.encode("."), [1]
    st = None if summ is None else np.asarray(_TOK.encode(summ) or _TOK.encode("."), dtype=np.uint16)
    return (np.asarray(toks, dtype=np.uint16), np.asarray(ends, dtype=np.int32), sentence_scores(sents), st)

def tokenizer_fingerprint(tok_path):
    from .tokenizer import BPETokenizer
    tok = BPETokenizer.load(tok_path)
    assert tok.vocab_size <= 65535, "uint16 token cache needs vocab <= 65535"
    return tok.fingerprint()

def cache_path(cache_dir, tok_path, max_docs=None, dev_size=300, dev_frac=0.05, split_seed=0):
    tag = tokenizer_fingerprint(tok_path) + f"_dev{dev_size}-{dev_frac}-{split_seed}" + (f"_n{max_docs}" if max_docs else "")
    return Path(cache_dir) / f"summ_{tag}.pkl"

def tokenise(items, tok_path, n_proc=4):
    if n_proc and n_proc > 1 and len(items) > 64:
        import gc
        import multiprocessing as mp
        gc.collect(); gc.freeze()
        try:
            with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(tok_path,)) as pool:
                return list(pool.imap(_encode_doc, items, chunksize=8))
        finally:
            gc.unfreeze()
    _init_worker(tok_path)
    return [_encode_doc(x) for x in items]

def _pack(docs, enc, source):
    return {"ids": [d["id"] for d in docs], "raw_ids": [d["raw_id"] for d in docs],
            "tokens": [e[0] for e in enc], "sent_end": [e[1] for e in enc], "sent_score": [e[2] for e in enc],
            "sum_tokens": None if enc and enc[0][3] is None else [e[3] for e in enc],
            "documents": [d["document"] for d in docs],
            "summaries": None if docs and docs[0]["summary"] is None else [d["summary"] for d in docs],
            "source": source}

def build_cache(data_dir, tok_path, cache_dir, dev_size=300, dev_frac=0.05, split_seed=0, n_proc=4, max_docs=None,
                log=print):
    import time
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    cp = cache_path(cache_dir, tok_path, max_docs, dev_size, dev_frac, split_seed)
    if cp.exists():
        return cp
    t0 = time.time()
    train = load_file(data_dir, "train", max_docs)
    test = load_file(data_dir, "test", max_docs)
    n_dev = min(dev_size, max(1, int(round(dev_frac * len(train))))) if len(train) > 20 else max(1, len(train) // 4)
    dv = dev_ids([d["id"] for d in train], n_dev, split_seed)
    parts = {"train": [d for d in train if d["id"] not in dv], "dev": [d for d in train if d["id"] in dv], "test": test}
    obj = {"split_seed": split_seed, "n_dev": n_dev}
    for split, docs in parts.items():
        enc = tokenise([(d["document"], d["summary"]) for d in docs], tok_path, n_proc)
        obj[split] = _pack(docs, enc, str(file_path(data_dir, "test" if split == "test" else "train")))
        nt = sum(len(e[0]) for e in enc)
        ns = sum(len(e[3]) for e in enc) if split != "test" else 0
        log(f"[data] {split}: {len(docs)} documents | {nt / 1e6:.1f}M document tokens"
            + (f" | {ns / 1e6:.2f}M summary tokens" if ns else ""))
    tmp = cp.with_suffix(f".{os.getpid()}.tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, cp)
    log(f"[data] tokenised in {time.time() - t0:.0f}s -> {cp.name}")
    return cp

def load_cached(data_dir, tok_path, cache_dir, max_docs=None, n_proc=4, dev_size=300, dev_frac=0.05, split_seed=0,
                log=print):
    cp = build_cache(data_dir, tok_path, cache_dir, dev_size, dev_frac, split_seed, n_proc, max_docs, log)
    with open(cp, "rb") as f:
        return pickle.load(f)

def subset(split, idx):
    return {k: ([v[i] for i in idx] if isinstance(v, list) else v) for k, v in split.items()}

def stats(data):
    out = {}
    for split in SPLITS:
        d = data[split]
        dl = np.array([len(t) for t in d["tokens"]])
        dw = np.array([len(x.split()) for x in d["documents"]])
        rec = {"documents": len(dl)}
        if len(dl):
            q = lambda a: {k: int(np.percentile(a, p)) for k, p in (("min", 0), ("p10", 10), ("p50", 50), ("p90", 90),
                                                                     ("max", 100))} | {"mean": int(a.mean())}
            rec["doc_words"], rec["doc_tokens"] = q(dw), q(dl)
            rec["tokens_per_word_doc"] = round(float(dl.sum() / max(1, dw.sum())), 3)
            rec["sentences_per_doc_p50"] = int(np.median([len(e) for e in d["sent_end"]]))
            for L in (1024, 2048, 4096, 8192):
                rec[f"doc_fits_in_{L}"] = round(float((dl <= L).mean()), 3)
        if d.get("sum_tokens"):
            sl = np.array([len(t) for t in d["sum_tokens"]])
            sw = np.array([len(x.split()) for x in d["summaries"]])
            rec["sum_words"], rec["sum_tokens"] = q(sw), q(sl)
            rec["compression_words_p50"] = round(float(np.median(dw / np.maximum(sw, 1))), 2)
            for L in (256, 384, 512, 640, 768, 1024):
                rec[f"sum_fits_in_{L}"] = round(float((sl <= L).mean()), 3)
            k = min(300, len(sw))
            for n in (1, 2, 3):
                rec[f"ref_novel_{n}gram"] = round(float(np.mean([novel_ngram_ratio(d["summaries"][i], d["documents"][i], n)
                                                                  for i in range(k)])), 4)
        out[split] = rec
    tr = set(data["train"]["ids"])
    fp = lambda s: {hashlib.sha1(t.encode()).hexdigest() for t in data[s]["documents"]}
    out["overlap"] = {"ids train&test": len(tr & set(data["test"]["ids"])), "ids train&dev": len(tr & set(data["dev"]["ids"])),
                      "identical_texts train&test": len(fp("train") & fp("test"))}
    return out

def ngrams(words, n):
    return [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]

def simple_words(text):
    return re.findall(r"\w+", text.lower())

def novel_ngram_ratio(summary, document, n):
    s = ngrams(simple_words(summary), n)
    if not s:
        return 0.0
    dset = set(ngrams(simple_words(document), n))
    return sum(g not in dset for g in s) / len(s)

def summary_source_positions(data, split="train", max_docs=500, n_bins=10):
    d = data[split]
    hist = np.zeros(n_bins)
    for i in range(min(max_docs, len(d["ids"]))):
        dsents = split_sentences(d["documents"][i])
        dsets = [set(content_words(s)) for s in dsents]
        if not dsets:
            continue
        for s in split_sentences(d["summaries"][i]):
            sw = set(content_words(s))
            if len(sw) < 3:
                continue
            ov = [len(sw & x) / len(sw) for x in dsets]
            j = int(np.argmax(ov))
            if ov[j] >= 0.3:
                hist[min(n_bins - 1, int(n_bins * (j + 0.5) / len(dsets)))] += 1
    return (hist / max(1, hist.sum())).round(4).tolist()

In [ ]:
%%writefile {WORK}/p6summ/context.py
"""Long-document handling: which tokens of a judgment the model reads (config section `context`).

A document of n tokens is reduced to at most `src_len` tokens (n <= src_len: the whole document):
    head       the first src_len tokens
    tail       the last src_len tokens
    headtail   the first head_frac * src_len tokens + the last (1 - head_frac) * src_len tokens
    uniform    `k` equal windows spread evenly over the document (sliding windows with a large stride)
    salient    importance-based selection: the first `lead` tokens, then whole sentences in decreasing salience
               (data.sentence_scores: TF-IDF similarity to the document centroid) until the budget is full,
               re-assembled in document order
Non-adjacent pieces are joined with a gap marker (" ...") so the model can see that text was skipped.

Encoder-decoder (B): the selected tokens are cut into chunks of the encoder length, every chunk is encoded
separately and the decoder cross-attends to all chunk encodings at once (Fusion-in-Decoder, Izacard & Grave 2021),
so B reads K x 768 tokens although its encoder was pretrained on 768.
"""
import numpy as np

def _spans_to_tokens(tokens, spans, gap_ids):
    out, prev_end = [], None
    for s, e in spans:
        if prev_end is not None and s != prev_end and gap_ids is not None and len(gap_ids):
            out.append(np.asarray(gap_ids, dtype=tokens.dtype))
        out.append(tokens[s:e])
        prev_end = e
    return np.concatenate(out) if out else tokens[:0]

def select_spans(n, budget, strategy="headtail", sent_end=None, sent_score=None, head_frac=0.5, k=4, lead=256,
                 gap_len=0):
    if n <= budget:
        return [(0, n)]
    if strategy == "head":
        return [(0, budget)]
    if strategy == "tail":
        return [(n - budget, n)]
    if strategy == "headtail":
        b = budget - gap_len
        h = int(round(b * head_frac))
        return [(0, h), (n - (b - h), n)] if 0 < h < b else ([(0, b)] if h >= b else [(n - b, n)])
    if strategy == "uniform":
        k = max(1, min(k, budget // 16))
        w = (budget - (k - 1) * gap_len) // k
        starts = np.linspace(0, n - w, k).round().astype(int)
        spans = []
        for s in starts:
            s = int(s)
            if spans and s < spans[-1][1]:
                s = spans[-1][1]
            if s + w <= n:
                spans.append((s, s + w))
        return _merge(spans)
    if strategy == "salient":
        assert sent_end is not None and sent_score is not None, "salient selection needs sentence boundaries"
        ends = np.asarray(sent_end, dtype=np.int64)
        starts = np.concatenate([[0], ends[:-1]])
        m = len(ends)
        chosen = np.zeros(m, dtype=bool)
        chosen[ends <= min(lead, budget)] = True
        used = int(ends[chosen].max()) if chosen.any() else 0
        for i in np.argsort(-np.asarray(sent_score), kind="stable"):
            if chosen[i]:
                continue
            ln = int(ends[i] - starts[i])
            nb = int(i > 0 and chosen[i - 1]) + int(i + 1 < m and chosen[i + 1])
            d_gap = (1 if chosen.any() else 0) - nb
            if used + ln + gap_len * d_gap <= budget:
                chosen[i] = True
                used += ln + gap_len * d_gap
        if not chosen.any():
            return [(0, budget)]
        spans = _merge([(int(starts[i]), int(ends[i])) for i in np.flatnonzero(chosen)])
        assert _cost(spans, gap_len) == used, (_cost(spans, gap_len), used)
        return spans
    raise ValueError(f"unknown context strategy {strategy}")

def _merge(spans):
    out = []
    for s, e in sorted(spans):
        if out and s <= out[-1][1]:
            out[-1] = (out[-1][0], max(out[-1][1], e))
        else:
            out.append((s, e))
    return out

def _cost(spans, gap_len):
    return sum(e - s for s, e in spans) + gap_len * sum(1 for a, b in zip(spans, spans[1:]) if b[0] != a[1])

def select_source(tokens, budget, strategy="headtail", sent_end=None, sent_score=None, head_frac=0.5, k=4, lead=256,
                  gap_ids=None):
    gl = len(gap_ids) if gap_ids is not None else 0
    spans = select_spans(len(tokens), budget, strategy, sent_end, sent_score, head_frac, k, lead, gl)
    out = _spans_to_tokens(np.asarray(tokens), spans, gap_ids)
    assert len(out) <= budget, (len(out), budget, strategy, spans)
    return out

def source_for(split, i, ccfg, budget, gap_ids):
    return select_source(split["tokens"][i], budget, ccfg["strategy"], split["sent_end"][i], split["sent_score"][i],
                         ccfg.get("head_frac", 0.5), ccfg.get("k", 4), ccfg.get("lead", 256), gap_ids)

def fid_chunks(src, chunk_len):
    return [src[i:i + chunk_len] for i in range(0, max(1, len(src)), chunk_len)] or [src]

In [ ]:
%%writefile {WORK}/p6summ/peft.py
"""Fine-tuning / parameter-efficient adaptation methods, applied in place to the blocks of a Phase-1 model.
(Phase-5 module; Phase 6 adds Pfeiffer adapters and AdaLoRA, and handles the encoder-decoder blocks of B, whose
decoder blocks have a self-attention, a cross-attention and an FFN.)

    full      every parameter trainable (embeddings / tied LM head included)
    partial   only the last `train_last_k` blocks (+ final norm) trainable (B: the last k decoder blocks)
    frozen    nothing trainable (used for the frozen KD teacher)
    lora      W x + (alpha/r) B A x                           (Hu et al., 2021)
              rslora=True: scale alpha/sqrt(r) instead of alpha/r  (Kalajdzievski, 2023, rank-stabilised LoRA)
    qlora     LoRA on top of a frozen base weight stored in 4-bit NormalFloat (NF4, blocks of 64 weights, one absmax
              scale per block; Dettmers et al., 2023). The fp32 base weight is deleted; the forward dequantises it
              on the fly. (No double quantisation of the scales and no paged optimiser: the model is small.)
    adalora   W x + P diag(lambda) Q x with an adaptive rank budget (Zhang et al., 2023): every triplet
              (P_i, lambda_i, Q_i) gets an importance score (EMA of |w * grad| times its uncertainty); the total number
              of non-zero lambdas is reduced from r * n_modules to target_r * n_modules on a cubic schedule by zeroing
              the least important lambdas; orthogonality penalty on P and Q. See AdaLoRAController.
    lorma     LoRMA+ : (I + (alpha/r) B A) W x                 (Bihany, Patel & Modi, 2025)
              LoRMApi: R_pi(B A) W x, rows of BA cyclically rotated (row i by i) -> full rank
    houlsby   Houlsby adapters: bottleneck after the attention output and after the FFN output: y + U act(D y)
              (Houlsby et al., 2019); B's decoder blocks get one after self-, one after cross-attention, one after FFN
    pfeiffer  Pfeiffer adapter: one bottleneck adapter per block, after the FFN output only (Pfeiffer et al., 2021)
    ia3       learned rescaling vectors: k, v (attention) and the FFN hidden activation   (Liu et al., 2022)
    bitfit    zero-initialised bias added to every block linear + RMSNorm gains (the Phase-1 linears have no biases)
    prompt    n learned vectors prepended to the input embeddings (Lester et al., 2021)   [A, C, D]
    prefix    prefix tuning (Li & Liang, 2021): every layer gets n learned virtual-token states; the layer's frozen
              norm + K/V projection turn them into extra keys/values that every real token attends to  [A, C]

New PEFT parameters are named lora_*, lorma_*, adapter_*, bitfit_*, ia3_*, soft_prompt or prefix_h so the trainer
can give them their own learning rate.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

NEW_PARAM_TAGS = ("lora_", "lorma_", "adapter_", "bitfit_", "ia3_", "soft_prompt", "prefix_h")

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0, rslora=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rslora else r)
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B)

NF4_CODES = (-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
             -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
             0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
             0.7229568362236023, 1.0)

def nf4_quantize(w, block=64):
    flat = w.detach().float().reshape(-1)
    assert flat.numel() % block == 0 and block % 2 == 0, f"{flat.numel()} weights not divisible into blocks of {block}"
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp(min=1e-12)
    codes = torch.tensor(NF4_CODES, dtype=torch.float32, device=flat.device)
    mids = (codes[1:] + codes[:-1]) / 2
    idx = torch.bucketize((blocks / absmax[:, None]).reshape(-1), mids).to(torch.uint8)
    return (idx[0::2] << 4) | idx[1::2], absmax

def nf4_dequantize(packed, absmax, shape, block=64):
    codes = torch.tensor(NF4_CODES, dtype=torch.float32, device=packed.device)
    idx = torch.stack([packed >> 4, packed & 15], 1).reshape(-1).long()
    return (codes[idx].view(-1, block) * absmax[:, None]).view(shape)

class QLoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0, block=64):
        super().__init__()
        assert base.bias is None, "Phase-1 linears have no bias"
        self.shape, self.block = tuple(base.weight.shape), block
        self.in_features, self.out_features = base.in_features, base.out_features
        self.weight_numel = base.weight.numel()
        packed, absmax = nf4_quantize(base.weight, block)
        self.register_buffer("nf4_packed", packed, persistent=False)
        self.register_buffer("nf4_absmax", absmax, persistent=False)
        self.scale = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def dequantized(self):
        return nf4_dequantize(self.nf4_packed, self.nf4_absmax, self.shape, self.block)

    def forward(self, x):
        h = F.linear(x, self.dequantized().to(x.dtype))
        return h + self.scale * F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)), self.lora_B.to(x.dtype))

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, variant="plus"):
        super().__init__()
        self.base, self.variant, self.scale = base, variant, alpha / r
        d = base.out_features
        self.d = d
        self.lorma_A = nn.Parameter(torch.empty(r, d))
        self.lorma_B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
            nn.init.zeros_(self.lorma_B)
        elif variant == "pi":
            nn.init.zeros_(self.lorma_A); self.lorma_A.data[0, 0] = 1.0
            nn.init.normal_(self.lorma_B, std=0.02); self.lorma_B.data[:, 0] = 1.0
        else:
            raise ValueError(variant)

    def rotated(self):
        M = self.lorma_B @ self.lorma_A
        d = self.d
        M2 = torch.cat([M, M], 1).contiguous()
        return M2.as_strided((d, d), (2 * d - 1, 1), storage_offset=d)

    def forward(self, x):
        h = self.base(x)
        if self.variant == "plus":
            return h + self.scale * F.linear(F.linear(h, self.lorma_A), self.lorma_B)
        return F.linear(h, self.rotated().to(h.dtype))

class HoulsbyAdapter(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        d = base.out_features
        self.adapter_down = nn.Linear(d, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.adapter_down.weight, std=0.02); nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight); nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.adapter_up(F.gelu(self.adapter_down(y)))

class IA3Linear(nn.Module):
    def __init__(self, base, mode="out", mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.ia3_l = nn.Parameter(torch.ones(n))
        self.register_buffer("ia3_mask", torch.ones(n) if mask is None else mask.float(), persistent=False)

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_l.to(x.dtype))
        s = 1 + (self.ia3_l - 1) * self.ia3_mask
        return self.base(x) * s.to(x.dtype)

class BitFitLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bitfit_bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bitfit_bias

class AdaLoRALinear(nn.Module):
    def __init__(self, base, r=12, alpha=24, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.lora_P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lora_E = nn.Parameter(torch.zeros(r))
        self.lora_Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(self.drop(x), self.lora_Q.to(x.dtype) * self.lora_E.to(x.dtype)[:, None])
        return self.base(x) + self.scale * F.linear(h, self.lora_P.to(x.dtype))

    def orth_penalty(self):
        P, Q = self.lora_P.float(), self.lora_Q.float()
        I = torch.eye(self.r, device=P.device)
        return ((P.t() @ P - I) ** 2).sum() + ((Q @ Q.t() - I) ** 2).sum()

class AdaLoRAController:
    def __init__(self, model, total_steps, target_r=4, init_warmup=0.1, final_warmup=0.2, delta_t=10, beta1=0.85,
                 beta2=0.85, orth_reg=0.1):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        assert self.mods, "no AdaLoRA modules in the model"
        self.T, self.dt, self.b1, self.b2, self.orth_reg = total_steps, max(1, delta_t), beta1, beta2, orth_reg
        self.t_init, self.t_final = int(init_warmup * total_steps), int(final_warmup * total_steps)
        self.b0 = sum(m.r for m in self.mods)
        self.bT = min(self.b0, target_r * len(self.mods))
        self.ipt, self.unc, self.final_mask = {}, {}, None
        self.budget = self.b0

    def orth_loss(self):
        return self.orth_reg * sum(m.orth_penalty() for m in self.mods) / len(self.mods)

    def schedule(self, t):
        if t < self.t_init:
            return self.b0
        if t >= self.T - self.t_final:
            return self.bT
        span = max(1, self.T - self.t_init - self.t_final)
        return int(self.bT + (self.b0 - self.bT) * (1 - (t - self.t_init) / span) ** 3)

    @torch.no_grad()
    def _score(self, p):
        if p.grad is None:
            return torch.zeros_like(p, dtype=torch.float32)
        ipt = (p.float() * p.grad.float()).abs()
        k = id(p)
        if k not in self.ipt:
            self.ipt[k], self.unc[k] = torch.zeros_like(ipt), torch.zeros_like(ipt)
        self.ipt[k].mul_(self.b1).add_(ipt, alpha=1 - self.b1)
        self.unc[k].mul_(self.b2).add_((ipt - self.ipt[k]).abs(), alpha=1 - self.b2)
        return self.ipt[k] * self.unc[k]

    @torch.no_grad()
    def update(self, step):
        grads = [p.grad for m in self.mods for p in (m.lora_P, m.lora_E, m.lora_Q) if p.grad is not None]
        if not all(bool(torch.isfinite(g).all()) for g in grads):
            return
        scores = [self._score(m.lora_E) + self._score(m.lora_P).mean(0) + self._score(m.lora_Q).mean(1) for m in self.mods]
        self.budget = self.schedule(step)
        if step >= self.T - self.t_final:
            if self.final_mask is None:
                self.final_mask = self._mask(scores, self.bT)
            for m, keep in zip(self.mods, self.final_mask):
                m.lora_E.masked_fill_(~keep, 0.0)
            return
        if step >= self.t_init and step % self.dt == 0:
            for m, keep in zip(self.mods, self._mask(scores, self.budget)):
                m.lora_E.masked_fill_(~keep, 0.0)

    def _mask(self, scores, budget):
        allv = torch.cat(scores)
        if budget >= len(allv):
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        thr = torch.kthvalue(allv, len(allv) - budget).values
        return [s > thr for s in scores]

    def ranks(self):
        return [int((m.lora_E != 0).sum()) for m in self.mods]

def _block_linears(blk, which=("attn", "ffn")):
    out = []
    if "attn" in which:
        for an in ("attn", "self_attn", "cross_attn"):
            a = getattr(blk, an, None)
            if a is None:
                continue
            for name in ("wq", "wkv", "qkv", "q", "kv", "proj"):
                if hasattr(a, name):
                    out.append((a, name, "attn_out" if name == "proj" else "attn_in"))
    if "ffn" in which:
        out.append((blk.ffn, "w12", "ffn_in"))
        out.append((blk.ffn, "proj", "ffn_out"))
    return out

def apply_method(model, cfg):
    name = cfg["name"]
    for p in model.parameters():
        p.requires_grad = name == "full"
    if name in ("full", "frozen"):
        return model
    blocks = model.peft_blocks()
    if name == "partial":
        k = cfg.get("train_last_k", 4)
        for blk in model.last_blocks(k):
            for p in blk.parameters():
                p.requires_grad = True
        for p in model.final_norm().parameters():
            p.requires_grad = True
        return model
    if name == "prefix":
        assert model.arch in ("A", "C"), "prefix tuning is implemented for the decoder (A) and prefix-LM (C) models"
        n = cfg.get("n_prefix", 16)
        g = torch.Generator().manual_seed(cfg.get("seed", 0))
        model.prefix_h = nn.ParameterList([
            nn.Parameter(model.tok_emb.weight[torch.randint(0, min(20000, model.tok_emb.num_embeddings), (n,), generator=g)].detach().clone())
            for _ in model.blocks])
        return model
    if name == "prompt":
        assert model.arch in ("A", "C", "D"), "prompt tuning is implemented for A, C and D"
        g = torch.Generator().manual_seed(cfg.get("seed", 0))
        model.soft_prompt = nn.Parameter(model.tok_emb.weight[torch.randint(0, min(20000, model.tok_emb.num_embeddings), (cfg.get("n_prompt", 20),),
                                                                            generator=g)].detach().clone())
        return model
    targets = cfg.get("targets", ["attn", "ffn"])
    for blk in blocks:
        if name in ("lora", "lorma", "qlora", "adalora"):
            for parent, attr, kind in _block_linears(blk, targets):
                base = getattr(parent, attr)
                if name == "qlora":
                    new = QLoRALinear(base, cfg.get("r", 8), cfg.get("alpha", 16), cfg.get("dropout", 0.05),
                                      cfg.get("nf4_block", 64))
                elif name == "lora":
                    new = LoRALinear(base, cfg.get("r", 8), cfg.get("alpha", 16), cfg.get("dropout", 0.05),
                                     cfg.get("rslora", False))
                elif name == "adalora":
                    new = AdaLoRALinear(base, cfg.get("r", 12), cfg.get("alpha", 24), cfg.get("dropout", 0.05))
                else:
                    if cfg.get("variant", "plus") == "pi" and base.out_features > cfg.get("pi_max_dim", 2048):
                        continue
                    new = LoRMALinear(base, cfg.get("r", 8), cfg.get("alpha", 16), cfg.get("variant", "plus"))
                setattr(parent, attr, new)
        elif name == "ia3":
            for an in ("attn", "self_attn", "cross_attn"):
                a = getattr(blk, an, None)
                if a is None:
                    continue
                if hasattr(a, "qkv"):
                    d = a.qkv.out_features // 3
                    a.qkv = IA3Linear(a.qkv, "out", torch.cat([torch.zeros(d), torch.ones(2 * d)]))
                elif hasattr(a, "wkv"):
                    a.wkv = IA3Linear(a.wkv, "out")
                elif hasattr(a, "kv"):
                    a.kv = IA3Linear(a.kv, "out")
            blk.ffn.proj = IA3Linear(blk.ffn.proj, "in")
        elif name in ("houlsby", "adapter", "pfeiffer"):
            kinds = ("ffn_out",) if name == "pfeiffer" else ("attn_out", "ffn_out")
            for parent, attr, kind in _block_linears(blk):
                if kind in kinds:
                    setattr(parent, attr, HoulsbyAdapter(getattr(parent, attr), cfg.get("bottleneck", 64)))
        elif name == "bitfit":
            for parent, attr, kind in _block_linears(blk):
                setattr(parent, attr, BitFitLinear(getattr(parent, attr)))
            for nm in ("norm1", "norm2", "norm3"):
                if hasattr(blk, nm):
                    getattr(blk, nm).weight.requires_grad = True
        else:
            raise ValueError(f"unknown method {name}")
    if name == "bitfit":
        for p in model.final_norm().parameters():
            p.requires_grad = True
    if cfg.get("train_norms", False):
        for blk in blocks:
            for nm in ("norm1", "norm2", "norm3"):
                if hasattr(blk, nm):
                    getattr(blk, nm).weight.requires_grad = True
        for p in model.final_norm().parameters():
            p.requires_grad = True
    return model

def is_new_param(name):
    return any(t in name for t in NEW_PARAM_TAGS)

def count_params(model):
    q = sum(m.weight_numel for m in model.modules() if isinstance(m, QLoRALinear))
    total = sum(p.numel() for p in model.parameters()) + q
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": train, "trainable_pct": round(100 * train / max(1, total), 3), "nf4_weights": q}

In [ ]:
%%writefile {WORK}/p6summ/model.py
"""Summarizer = a Phase-1 language model (+ PEFT) used as a conditional generator: document -> summary.

How each Phase-1 architecture is turned into a summariser (the formulation matches how it was pretrained):

  A  decoder-only (LLaMA-style, GQA)     [document ; "SUMMARY:" ; summary <eot>]   causal LM, loss on the summary
  D  efficient decoder (sliding window)  same sequence, sliding-window causal attention (window 256)
  C  hybrid prefix-LM (UniLM-style)      same sequence; the document + separator is the *prefix* (bidirectional,
                                         segment 0), the summary is the causal suffix (segment 1): exactly the
                                         prefix-LM objective of Phase 1
  B  encoder-decoder                     encoder reads the document chunks (Fusion-in-Decoder: every chunk of <= 768
                                         tokens is encoded separately, the decoder cross-attends to all of them),
                                         decoder generates <bos> summary <eot>

Positions: A and C were pretrained with 1024 positions, B's decoder with 384. When a run needs longer sequences,
the RoPE table is rebuilt with *position interpolation* (Chen et al., 2023): position p uses the rotation angle of
p / s with s = needed / pretrained length, so all angles stay inside the pretrained range ("rope_scale": "auto").
D's RoPE table already has 8192 positions and its attention only spans 256 tokens, so it needs no interpolation.

Attention is recomputed here from each block's own (possibly PEFT-wrapped) projection layers, with
  * explicit padding masks and per-sample positions, so a batch of prompts of different length can be generated
    together with a KV cache (prompts are prefilled one by one, caches are left-padded, padded keys masked out);
  * prefix-tuning keys/values and prompt-tuning vectors.
tests/test_pipeline.py checks that the training forward reproduces the original Phase-1 forward exactly and that
cached generation reproduces the full forward.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from .phase1_models import MODEL_CLASSES

ARCH_OF_CLASS = {"LlamaLM": "A", "EncDecLM": "B", "HybridLM": "C", "EfficientLM": "D"}

def _clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        for p in ("_orig_mod.", "module."):
            while k.startswith(p):
                k = k[len(p):]
        out[k] = v
    return out

def load_phase1(ckpt_path, random_init=False):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cls_name = ck.get("model_class")
    kw = dict(ck["model_kw"])
    if cls_name is None:
        cls_name = ("EncDecLM" if "enc_block" in kw else "HybridLM" if "n_segments" in kw
                    else "EfficientLM" if "window" in kw else "LlamaLM")
    model = MODEL_CLASSES[cls_name](**kw)
    if not random_init:
        missing, unexpected = model.load_state_dict(_clean_state_dict(ck["model"]), strict=False)
        missing = [m for m in missing if not m.endswith("lm_head.weight")]
        assert not missing and not unexpected, f"checkpoint mismatch: missing={missing} unexpected={unexpected}"
    return model, cls_name, kw

class ScaledRope(nn.Module):
    def __init__(self, head_dim, length, theta=10000.0, scale=1.0):
        super().__init__()
        inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        pos = torch.arange(length).float()
        freqs = torch.outer(pos / scale if scale != 1.0 else pos, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)
        self.length, self.scale = length, scale

    def forward(self, x, pos=0):
        T = x.size(-2)
        if isinstance(pos, int):
            assert pos + T <= self.length, f"position {pos + T} > RoPE table {self.length}"
            cos, sin = self.cos[pos:pos + T], self.sin[pos:pos + T]
        else:
            assert int(pos.max()) < self.length, f"position {int(pos.max())} >= RoPE table {self.length}"
            cos, sin = self.cos[pos][:, None], self.sin[pos][:, None]
        x1, x2 = x.float().chunk(2, dim=-1)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

def project_qkv(attn, x):
    B, T, _ = x.shape
    H, hd = attn.n_head, attn.hd
    if hasattr(attn, "qkv"):
        q, k, v = attn.qkv(x).view(B, T, 3, H, hd).permute(2, 0, 3, 1, 4)
        return q, k, v
    q = attn.wq(x).view(B, T, H, hd).transpose(1, 2)
    k, v = attn.wkv(x).view(B, T, 2, attn.n_kv, hd).permute(2, 0, 3, 1, 4)
    return q, k, v

def expand_kv(t, H):
    return t if t.size(1) == H else t.repeat_interleave(H // t.size(1), dim=1)

def prefix_kv(attn, h):
    P = h.size(0)
    H, hd = attn.n_head, attn.hd
    if hasattr(attn, "qkv"):
        _, k, v = attn.qkv(h).view(P, 3, H, hd).permute(1, 2, 0, 3)
    else:
        k, v = attn.wkv(h).view(P, 2, attn.n_kv, hd).permute(1, 2, 0, 3)
        k, v = expand_kv(k[None], H)[0], expand_kv(v[None], H)[0]
    return k, v

def self_attention(attn, x, rope, pos=0, mask=None, is_causal=False, past=None, prefix=None, window=None,
                   blocked=False):
    B, T, _ = x.shape
    q, k, v = project_qkv(attn, x)
    q, k = rope(q, pos), rope(k, pos)
    if past is not None:
        k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
    if window is not None and k.size(2) > window:
        k, v = k[:, :, -window:], v[:, :, -window:]
    if blocked:
        y = attn._blocked(q, k, v)
    else:
        kk, vv = expand_kv(k, attn.n_head), expand_kv(v, attn.n_head)
        if prefix is not None:
            pk, pv = prefix
            kk = torch.cat([pk[None].expand(B, -1, -1, -1).to(kk.dtype), kk], 2)
            vv = torch.cat([pv[None].expand(B, -1, -1, -1).to(vv.dtype), vv], 2)
        y = F.scaled_dot_product_attention(q, kk, vv, attn_mask=mask, is_causal=is_causal and mask is None,
                                           dropout_p=attn.dropout if attn.training else 0.0)
    return attn.resid_drop(attn.proj(y.transpose(1, 2).reshape(B, T, -1))), (k, v)

def memory_kv(attn, memory):
    B, S, _ = memory.shape
    k, v = attn.kv(memory).view(B, S, 2, attn.n_head, attn.hd).permute(2, 0, 3, 1, 4)
    return k, v

def cross_attention(attn, x, kv, mask):
    B, T, _ = x.shape
    q = attn.q(x).view(B, T, attn.n_head, attn.hd).transpose(1, 2)
    y = F.scaled_dot_product_attention(q, kv[0], kv[1], attn_mask=mask,
                                       dropout_p=attn.dropout if attn.training else 0.0)
    return attn.resid_drop(attn.proj(y.transpose(1, 2).reshape(B, T, -1)))

class Summarizer(nn.Module):
    def __init__(self, ckpt, keep_layers=None, grad_ckpt=False, random_init=False):
        super().__init__()
        m, cls_name, kw = load_phase1(ckpt, random_init)
        self.arch, self.cls_name, self.kw = ARCH_OF_CLASS[cls_name], cls_name, kw
        self.tok_emb, self.lm_head, self.drop = m.tok_emb, m.lm_head, m.drop
        self.theta = kw.get("rope_theta", 10000.0)
        self.grad_ckpt = grad_ckpt
        self.prefix_h = None
        self.soft_prompt = None
        self.bos_id = None
        if self.arch == "B":
            n_e, n_d = len(m.enc_blocks), len(m.dec_blocks)
            self.phase1_layers = n_e + n_d
            self.enc_blocks = nn.ModuleList(list(m.enc_blocks)[:keep_layers] if keep_layers else list(m.enc_blocks))
            self.dec_blocks = nn.ModuleList(list(m.dec_blocks)[:keep_layers] if keep_layers else list(m.dec_blocks))
            self.enc_norm, self.dec_norm = m.enc_norm, m.dec_norm
            self.hd = kw["d_model"] // kw["n_head"]
            self.enc_len, self.dec_pretrained = kw["enc_block"], kw["dec_block"]
        else:
            self.phase1_layers = len(m.blocks)
            self.blocks = nn.ModuleList(list(m.blocks)[:keep_layers] if keep_layers else list(m.blocks))
            self.norm_f = m.norm_f
            self.hd = kw["d_model"] // kw["n_head"]
            self.seg_emb = m.seg_emb if self.arch == "C" else None
            self.window = m.window if self.arch == "D" else None
            self.pretrained_len = m.max_pos if self.arch == "D" else kw["block_size"]
        self.rope = self.enc_rope = self.dec_rope = None
        del m

    def peft_blocks(self):
        return list(self.enc_blocks) + list(self.dec_blocks) if self.arch == "B" else list(self.blocks)

    def last_blocks(self, k):
        return list(self.dec_blocks)[-k:] if self.arch == "B" else list(self.blocks)[-k:]

    def final_norm(self):
        return self.dec_norm if self.arch == "B" else self.norm_f

    def n_layers(self):
        return len(self.peft_blocks())

    def set_lengths(self, total_len, rope_scale="auto", table_len=None):
        n_p = 0 if self.soft_prompt is None else self.soft_prompt.size(0)
        need = total_len + n_p
        table = max(need, (table_len or 0) + n_p)
        dev = self.tok_emb.weight.device
        base = self.dec_pretrained if self.arch == "B" else self.pretrained_len
        s = max(1.0, need / base) if rope_scale == "auto" else float(rope_scale)
        assert s >= 1.0, "rope_scale must be >= 1"
        if self.arch == "B":
            self.enc_rope = ScaledRope(self.hd, self.enc_len, self.theta).to(dev)
            self.dec_rope = ScaledRope(self.hd, max(table, base), self.theta, s).to(dev)
            return {"dec_positions": max(table, base), "dec_rope_scale": round(s, 4)}
        self.rope = ScaledRope(self.hd, max(table, base), self.theta, s).to(dev)
        return {"positions": max(table, base), "rope_scale": round(s, 4)}

    def logits(self, h):
        return self.lm_head(h)

    def _prefix(self, li, dtype):
        if self.prefix_h is None:
            return None
        blk = self.blocks[li]
        return prefix_kv(blk.attn, blk.norm1(self.prefix_h[li].to(dtype)))

    def _embed(self, ids, seg=None):
        x = self.tok_emb(ids)
        if self.seg_emb is not None:
            x = x + (self.seg_emb(seg) if torch.is_tensor(seg) else self.seg_emb.weight[seg])
        return x

    def _train_mask(self, prefix_len, T, n_pre, device):
        if self.arch == "D":
            assert n_pre == 0, "prefix tuning is not implemented for D"
            return None, True, True
        if self.arch == "C" and prefix_len is not None and not bool((prefix_len == 0).all()):
            pos = torch.arange(T, device=device)
            m = (pos[None, None, :] < prefix_len[:, None, None]) | (pos[None, :, None] >= pos[None, None, :])
            m = m[:, None]
        elif n_pre:
            m = torch.ones(T, T, dtype=torch.bool, device=device).tril()[None, None]
        else:
            return None, True, False
        if n_pre:
            m = torch.cat([torch.ones(*m.shape[:-1], n_pre, dtype=torch.bool, device=device), m], -1)
        return m, False, False

    def _dec_block(self, li, x, mask, causal, blocked):
        blk = self.blocks[li]
        a, _ = self_attention(blk.attn, blk.norm1(x), self.rope, 0, mask, causal,
                              prefix=self._prefix(li, x.dtype), blocked=blocked)
        x = x + a
        return x + blk.ffn(blk.norm2(x))

    def forward_dec(self, ids, labels, prefix_len=None):
        B, T = ids.shape
        n_p = 0
        if self.arch == "C":
            pl = prefix_len if prefix_len is not None else torch.zeros(B, dtype=torch.long, device=ids.device)
            seg = (torch.arange(T, device=ids.device)[None, :] >= pl[:, None]).long()
            x = self._embed(ids, seg)
        else:
            x = self._embed(ids)
        if self.soft_prompt is not None:
            n_p = self.soft_prompt.size(0)
            sp = self.soft_prompt[None].expand(B, -1, -1).to(x.dtype)
            if self.seg_emb is not None:
                sp = sp + self.seg_emb.weight[0 if prefix_len is not None and bool((prefix_len > 0).all()) else 1]
            x = torch.cat([sp, x], 1)
            labels = torch.cat([torch.full((B, n_p), -100, dtype=labels.dtype, device=labels.device), labels], 1)
            if prefix_len is not None:
                prefix_len = torch.where(prefix_len > 0, prefix_len + n_p, prefix_len)
            T = T + n_p
        x = self.drop(x)
        n_pre = 0 if self.prefix_h is None else self.prefix_h[0].size(0)
        mask, causal, blocked = self._train_mask(prefix_len if self.arch == "C" else None, T, n_pre, ids.device)
        for li in range(len(self.blocks)):
            if self.grad_ckpt and self.training:
                x = checkpoint(self._dec_block, li, x, mask, causal, blocked, use_reentrant=False)
            else:
                x = self._dec_block(li, x, mask, causal, blocked)
        sel = labels != -100
        return self.logits(self.norm_f(x[sel])), labels[sel]

    def encode(self, ids, keep):
        x = self.drop(self.tok_emb(ids))
        mask = None if bool(keep.all()) else keep[:, None, None, :]
        for blk in self.enc_blocks:
            if self.grad_ckpt and self.training:
                x = checkpoint(self._enc_block, blk, x, mask, use_reentrant=False)
            else:
                x = self._enc_block(blk, x, mask)
        return self.enc_norm(x)

    def _enc_block(self, blk, x, mask):
        x = x + self_attention(blk.attn, blk.norm1(x), self.enc_rope, 0, mask)[0]
        return x + blk.ffn(blk.norm2(x))

    @staticmethod
    def fid_memory(enc, keep, doc, slot, n_docs, k_max):
        N, S, C = enc.shape
        mem = enc.new_zeros(n_docs, k_max, S, C)
        mm = torch.zeros(n_docs, k_max, S, dtype=torch.bool, device=enc.device)
        mem[doc, slot] = enc
        mm[doc, slot] = keep
        return mem.view(n_docs, k_max * S, C), mm.view(n_docs, k_max * S)

    def _dec_block_b(self, blk, x, memory, mm4):
        x = x + self_attention(blk.self_attn, blk.norm1(x), self.dec_rope, 0, None, True)[0]
        x = x + cross_attention(blk.cross_attn, blk.norm2(x), memory_kv(blk.cross_attn, memory), mm4)
        return x + blk.ffn(blk.norm3(x))

    def forward_encdec(self, src, keep, doc, slot, n_docs, k_max, dec_in, labels):
        memory, mm = self.fid_memory(self.encode(src, keep), keep, doc, slot, n_docs, k_max)
        mm4 = None if bool(mm.all()) else mm[:, None, None, :]
        x = self.drop(self.tok_emb(dec_in))
        for blk in self.dec_blocks:
            if self.grad_ckpt and self.training:
                x = checkpoint(self._dec_block_b, blk, x, memory, mm4, use_reentrant=False)
            else:
                x = self._dec_block_b(blk, x, memory, mm4)
        sel = labels != -100
        return self.logits(self.dec_norm(x[sel])), labels[sel]

    def forward(self, batch):
        if self.arch == "B":
            return self.forward_encdec(batch["src"], batch["keep"], batch["doc"], batch["slot"], batch["n_docs"],
                                       batch["k_max"], batch["dec_in"], batch["labels"])
        return self.forward_dec(batch["ids"], batch["labels"], batch.get("prefix_len"))

    @torch.no_grad()
    def prefill(self, inputs):
        if self.arch == "B":
            return self._prefill_b(inputs)
        dev = self.tok_emb.weight.device
        L = len(self.blocks)
        ks, vs, lens, last = [[] for _ in range(L)], [[] for _ in range(L)], [], []
        for p in inputs:
            ids = p.to(dev)[None]
            x = self._embed(ids, 0)
            if self.soft_prompt is not None:
                sp = self.soft_prompt[None].to(x.dtype)
                if self.seg_emb is not None:
                    sp = sp + self.seg_emb.weight[0]
                x = torch.cat([sp, x], 1)
            T = x.size(1)
            for li, blk in enumerate(self.blocks):
                pre = self._prefix(li, x.dtype)
                if self.arch == "A":
                    mask = None if pre is None else torch.cat(
                        [torch.ones(T, pre[0].size(1), dtype=torch.bool, device=dev),
                         torch.ones(T, T, dtype=torch.bool, device=dev).tril()], 1)[None, None]
                    a, (k, v) = self_attention(blk.attn, blk.norm1(x), self.rope, 0, mask, pre is None, prefix=pre)
                elif self.arch == "C":
                    a, (k, v) = self_attention(blk.attn, blk.norm1(x), self.rope, 0, None, False, prefix=pre)
                else:
                    a, (k, v) = self_attention(blk.attn, blk.norm1(x), self.rope, 0, blocked=True)
                    k, v = k[:, :, -self.window:], v[:, :, -self.window:]
                x = x + a
                x = x + blk.ffn(blk.norm2(x))
                ks[li].append(k[0]); vs[li].append(v[0])
            lens.append(T)
            last.append(self.logits(self.norm_f(x[:, -1]))[0])
        Tk = max(k.size(1) for k in ks[0])
        n = len(inputs)
        mask = torch.zeros(n, Tk, dtype=torch.bool, device=dev)
        state = {"k": [], "v": [], "pos": torch.tensor(lens, dtype=torch.long, device=dev)}
        for li in range(L):
            K = ks[li][0].new_zeros(n, ks[li][0].size(0), Tk, self.hd)
            V = torch.zeros_like(K)
            for b in range(n):
                t = ks[li][b].size(1)
                K[b, :, Tk - t:], V[b, :, Tk - t:] = ks[li][b], vs[li][b]
                if li == 0:
                    mask[b, Tk - t:] = True
            state["k"].append(K); state["v"].append(V)
        state["mask"] = mask
        state["logits"] = torch.stack(last).float()
        return state

    @torch.no_grad()
    def step(self, state, tok):
        if self.arch == "B":
            return self._step_b(state, tok)
        B = tok.size(0)
        x = self._embed(tok[:, None], 1)
        pos = state["pos"][:, None]
        mask = torch.cat([state["mask"], torch.ones(B, 1, dtype=torch.bool, device=tok.device)], 1)
        if self.window is not None:
            mask = mask[:, -self.window:]
        m4 = mask[:, None, None, :]
        for li, blk in enumerate(self.blocks):
            pre = self._prefix(li, x.dtype)
            mm = m4 if pre is None else torch.cat(
                [torch.ones(B, 1, 1, pre[0].size(1), dtype=torch.bool, device=tok.device), m4], -1)
            a, (k, v) = self_attention(blk.attn, blk.norm1(x), self.rope, pos, mm,
                                       past=(state["k"][li], state["v"][li]), prefix=pre, window=self.window)
            state["k"][li], state["v"][li] = k, v
            x = x + a
            x = x + blk.ffn(blk.norm2(x))
        state["mask"] = mask
        state["pos"] = state["pos"] + 1
        return self.logits(self.norm_f(x[:, 0])).float()

    def _prefill_b(self, inputs):
        dev = self.tok_emb.weight.device
        n = len(inputs)
        chunks = [(b, k, c) for b, doc in enumerate(inputs) for k, c in enumerate(doc)]
        S = max(len(c) for _, _, c in chunks)
        k_max = max(len(doc) for doc in inputs)
        ids = torch.full((len(chunks), S), self.kw.get("pad_id") or 0, dtype=torch.long, device=dev)
        keep = torch.zeros(len(chunks), S, dtype=torch.bool, device=dev)
        doc = torch.tensor([b for b, _, _ in chunks], device=dev)
        slot = torch.tensor([k for _, k, _ in chunks], device=dev)
        for r, (_, _, c) in enumerate(chunks):
            ids[r, :len(c)] = c.to(dev)
            keep[r, :len(c)] = True
        memory, mm = self.fid_memory(self.encode(ids, keep), keep, doc, slot, n, k_max)
        state = {"self_k": [None] * len(self.dec_blocks), "self_v": [None] * len(self.dec_blocks),
                 "cross_k": [], "cross_v": [], "mem_mask": mm, "pos": torch.zeros(n, dtype=torch.long, device=dev)}
        for blk in self.dec_blocks:
            k, v = memory_kv(blk.cross_attn, memory)
            state["cross_k"].append(k); state["cross_v"].append(v)
        bos = torch.full((n,), self.bos_id, dtype=torch.long, device=dev)
        state["logits"] = self._step_b(state, bos)
        return state

    def _step_b(self, state, tok):
        x = self.tok_emb(tok[:, None])
        pos = state["pos"][:, None]
        mm4 = state["mem_mask"][:, None, None, :]
        for li, blk in enumerate(self.dec_blocks):
            past = None if state["self_k"][li] is None else (state["self_k"][li], state["self_v"][li])
            a, (k, v) = self_attention(blk.self_attn, blk.norm1(x), self.dec_rope, pos, None, past=past)
            state["self_k"][li], state["self_v"][li] = k, v
            x = x + a
            x = x + cross_attention(blk.cross_attn, blk.norm2(x), (state["cross_k"][li], state["cross_v"][li]), mm4)
            x = x + blk.ffn(blk.norm3(x))
        state["pos"] = state["pos"] + 1
        return self.logits(self.dec_norm(x[:, 0])).float()

def map_state(state, fn):
    out = {}
    for k, v in state.items():
        if torch.is_tensor(v):
            out[k] = fn(v)
        elif isinstance(v, list):
            out[k] = [None if t is None else fn(t) for t in v]
        else:
            out[k] = v
    return out

def seq_loss(logits, labels, teacher_logits=None, kd_alpha=0.5, kd_T=2.0):
    ce = F.cross_entropy(logits.float(), labels)
    if teacher_logits is None:
        return ce, {"ce": float(ce.detach())}
    T = kd_T
    kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.log_softmax(teacher_logits.float() / T, -1),
                  reduction="batchmean", log_target=True) * T * T
    return (1 - kd_alpha) * ce + kd_alpha * kl, {"ce": float(ce.detach()), "kd": float(kl.detach())}

def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

In [ ]:
%%writefile {WORK}/p6summ/generate.py
"""Decoding (config section `gen`): greedy, beam search or sampling, all with a KV cache.

Logit processing at every step (in this order):
    special tokens <pad> <bos> <sep> <mask> are never generated; <eot> (end of summary) is blocked until
    `min_new` tokens exist (length control)
    repetition penalty  logit / p (positive) or * p (negative) for tokens among the last `rep_window` generated
    no-repeat n-gram    a token that would complete an n-gram already present in the hypothesis gets -inf
                        (vectorised: compare the last n-1 tokens with every earlier (n-1)-gram)
then greedy = argmax, sample = temperature / top-k / top-p, beam = `beams` hypotheses per document scored by
sum of log-probabilities / length^length_penalty.
Generation stops at <eot> or after `max_new` tokens.
"""
import time

import numpy as np
import torch

from .model import map_state

GEN_DEFAULTS = {"method": "greedy", "beams": 4, "max_new": 640, "min_new": 64, "no_repeat_ngram": 4,
                "rep_penalty": 1.0, "rep_window": 512, "length_penalty": 1.0, "temperature": 0.7, "top_k": 50,
                "top_p": 0.9, "batch_size": 16, "seed": 0}

def process_logits(logits, hist, step, g, sp):
    logits[:, sp["banned"]] = -float("inf")
    if step < g["min_new"]:
        logits[:, sp["eot"]] = -float("inf")
    t = hist.size(1)
    if t and g.get("rep_penalty", 1.0) != 1.0:
        ctx = hist[:, -g["rep_window"]:]
        sc = torch.gather(logits, 1, ctx)
        logits.scatter_(1, ctx, torch.where(sc > 0, sc / g["rep_penalty"], sc * g["rep_penalty"]))
    n = g.get("no_repeat_ngram", 0)
    if n and t >= n:
        win = hist.unfold(1, n - 1, 1)
        match = (win[:, :-1] == hist[:, None, t - n + 1:]).all(-1)
        nxt = hist[:, n - 1:]
        logits.scatter_(1, torch.where(match, nxt, sp["banned"][0]), -float("inf"))
    return logits

def _sample(logits, g, gen):
    logits = logits / max(g["temperature"], 1e-5)
    if g.get("top_k"):
        v, _ = torch.topk(logits, min(g["top_k"], logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float("inf")
    if g.get("top_p") and g["top_p"] < 1.0:
        sl, si = torch.sort(logits, descending=True)
        probs = torch.softmax(sl, -1)
        sl[probs.cumsum(-1) - probs > g["top_p"]] = -float("inf")
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    return torch.multinomial(torch.softmax(logits, -1), 1, generator=gen)[:, 0]

@torch.no_grad()
def _greedy_or_sample(model, state, g, sp, gen):
    logits = state.pop("logits")
    R = logits.size(0)
    dev = logits.device
    hist = torch.zeros(R, 0, dtype=torch.long, device=dev)
    done = torch.zeros(R, dtype=torch.bool, device=dev)
    for step in range(g["max_new"]):
        lg = process_logits(logits, hist, step, g, sp)
        tok = lg.argmax(-1) if g["method"] == "greedy" else _sample(lg, g, gen)
        tok = torch.where(done, torch.full_like(tok, sp["eot"]), tok)
        hist = torch.cat([hist, tok[:, None]], 1)
        done |= tok == sp["eot"]
        if bool(done.all()):
            break
        logits = model.step(state, tok)
    out = []
    for row in hist.tolist():
        out.append(row[:row.index(sp["eot"])] if sp["eot"] in row else row)
    return out

@torch.no_grad()
def _beam(model, state, g, sp):
    nb, lp = g["beams"], g.get("length_penalty", 1.0)
    logits = state.pop("logits")
    B, V = logits.shape
    dev = logits.device
    rep = torch.arange(B, device=dev).repeat_interleave(nb)
    state = map_state(state, lambda t: t.index_select(0, rep))
    logits = logits.index_select(0, rep)
    scores = torch.full((B, nb), -float("inf"), device=dev)
    scores[:, 0] = 0.0
    hist = torch.zeros(B * nb, 0, dtype=torch.long, device=dev)
    finished = [[] for _ in range(B)]
    done = [False] * B
    for step in range(g["max_new"]):
        lp_ = torch.log_softmax(process_logits(logits, hist, step, g, sp), -1)
        cand = (scores.view(-1, 1) + lp_).view(B, nb * V)
        top_s, top_i = cand.topk(2 * nb, dim=1)
        top_s, top_i = top_s.tolist(), top_i.tolist()
        new_scores = torch.full((B, nb), -float("inf"), device=dev)
        src = torch.zeros(B * nb, dtype=torch.long, device=dev)
        toks = torch.full((B * nb,), sp["eot"], dtype=torch.long, device=dev)
        src_l, tok_l, sc_l = [], [], []
        for b in range(B):
            rows, toks_b, scs = [], [], []
            if not done[b]:
                for s, i in zip(top_s[b], top_i[b]):
                    if s == -float("inf"):
                        break
                    beam, tok = divmod(i, V)
                    r = b * nb + beam
                    if tok == sp["eot"]:
                        finished[b].append((s / (step + 1) ** lp, hist[r].tolist()))
                    else:
                        rows.append(r); toks_b.append(tok); scs.append(s)
                    if len(rows) == nb:
                        break
                finished[b] = sorted(finished[b], key=lambda x: -x[0])[:nb]
                if len(finished[b]) >= nb and (not scs or max(scs) / (step + 1) ** lp <= finished[b][-1][0]):
                    done[b] = True
            while len(rows) < nb:
                rows.append(b * nb); toks_b.append(sp["eot"]); scs.append(-float("inf"))
            src_l += rows; tok_l += toks_b; sc_l += scs
        src = torch.tensor(src_l, device=dev)
        toks = torch.tensor(tok_l, device=dev)
        scores = torch.tensor(sc_l, device=dev).view(B, nb)
        hist = torch.cat([hist.index_select(0, src), toks[:, None]], 1)
        if all(done):
            break
        state = map_state(state, lambda t: t.index_select(0, src))
        logits = model.step(state, toks)
    out = []
    for b in range(B):
        if not done[b]:
            for k in range(nb):
                s = float(scores[b, k])
                if s > -float("inf"):
                    row = hist[b * nb + k].tolist()
                    finished[b].append((s / max(1, len(row)) ** lp, row))
        best = max(finished[b], key=lambda x: x[0])[1] if finished[b] else []
        out.append([t for t in best if t != sp["eot"]])
    return out

def special_ids(tok):
    return {"eot": tok.special_to_id["<|endoftext|>"],
            "banned": [tok.pad_id, tok.bos_id, tok.special_to_id["<|sep|>"], tok.mask_id]}

@torch.no_grad()
def generate(model, inputs, gcfg, tok, device="cuda", log=None, amp_dtype=None):
    g = {**GEN_DEFAULTS, **(gcfg or {})}
    sp = special_ids(tok)
    sp["banned"] = torch.tensor(sp["banned"], device=device)
    model.eval()
    size = (lambda x: sum(len(c) for c in x)) if model.arch == "B" else len
    order = np.argsort([-size(x) for x in inputs], kind="stable")
    bs = max(1, g["batch_size"] // (g["beams"] if g["method"] == "beam" else 1))
    out = [None] * len(inputs)
    gen = torch.Generator(device=device).manual_seed(g.get("seed", 0)) if g["method"] == "sample" else None
    t0, n_tok = time.time(), 0
    for k in range(0, len(order), bs):
        idx = order[k:k + bs]
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None and device == "cuda"):
            state = model.prefill([inputs[i] for i in idx])
            res = _beam(model, state, g, sp) if g["method"] == "beam" else _greedy_or_sample(model, state, g, sp, gen)
        for i, r in zip(idx, res):
            out[i] = r
            n_tok += len(r)
        if log and (k // bs) % 10 == 0:
            log(f"[gen] {min(k + bs, len(order))}/{len(order)} documents | {time.time() - t0:.0f}s")
    dt = time.time() - t0
    return out, {"seconds": round(dt, 1), "docs_per_sec": round(len(inputs) / max(dt, 1e-6), 3),
                 "tokens_per_sec": round(n_tok / max(dt, 1e-6), 1), "mean_new_tokens": round(n_tok / max(1, len(inputs)), 1)}

In [ ]:
%%writefile {WORK}/p6summ/metrics.py
"""Summarisation metrics (Codabench: BERTScore F1 primary; ROUGE-1 F1 and ROUGE-L F1 secondary) and the analysis
measures used in the report.

ROUGE: Google's `rouge_score` package (rouge1 / rouge2 / rougeL, Porter stemmer) when it is installed, otherwise an
       identical-in-spirit re-implementation (lower-case alphanumeric tokens, Porter-free) - the notebook installs the
       package, tests/test_pipeline.py checks the fallback against it.
BERTScore: the `bert_score` package (default English model roberta-large, layer 17, no IDF, no baseline rescaling),
       only used to *score* outputs locally the way the leaderboard does. Note: bert_score truncates every text to
       the scoring model's maximum length (512 sub-words), so words beyond roughly the first ~400 of a summary do not
       change BERTScore (they do change ROUGE) - the length analysis measures this directly.
Analysis measures:
    redundancy     share of repeated 3-grams inside a summary; share of repeated sentences
    novelty        share of summary n-grams that do not occur in the source document (abstractiveness)
    legal facts    numbers, dates, years, "Section/Article/Rule/Order N", "... Act, YYYY", case-number patterns:
                   precision = share of generated facts that occur in the source (hallucination proxy),
                   recall    = share of the reference summary's facts that the generated summary contains (coverage)
"""
import re
from collections import Counter

import numpy as np

_TOK_RE = re.compile(r"[a-z0-9]+")

def _tokens(text):
    return _TOK_RE.findall(text.lower())

def _ngrams(t, n):
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))

def _prf(overlap, n_pred, n_ref):
    p = overlap / n_pred if n_pred else 0.0
    r = overlap / n_ref if n_ref else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def _lcs(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rouge_fallback(pred, ref, lcs=True):
    p, r = _tokens(pred), _tokens(ref)
    out = {}
    for n in (1, 2):
        a, b = _ngrams(p, n), _ngrams(r, n)
        out[f"rouge{n}"] = _prf(sum((a & b).values()), max(0, len(p) - n + 1), max(0, len(r) - n + 1))
    if lcs:
        out["rougeL"] = _prf(_lcs(p, r), len(p), len(r))
    return out

_SCORER = None

def rouge_scores(preds, refs, use_package=True):
    global _SCORER
    scorer = None
    if use_package:
        try:
            if _SCORER is None:
                from rouge_score import rouge_scorer
                _SCORER = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
            scorer = _SCORER
        except ImportError:
            scorer = None
    per = {k: [] for k in ("rouge1", "rouge2", "rougeL")}
    for p, r in zip(preds, refs):
        if scorer is not None:
            s = scorer.score(r, p)
            for k in per:
                per[k].append((s[k].precision, s[k].recall, s[k].fmeasure))
        else:
            s = rouge_fallback(p, r)
            for k in per:
                per[k].append(s[k])
    out = {"rouge_impl": "rouge_score" if scorer is not None else "fallback"}
    for k, v in per.items():
        a = np.array(v) if v else np.zeros((0, 3))
        out[f"{k}_p"], out[f"{k}_r"], out[f"{k}_f"] = (float(a[:, i].mean()) if len(a) else 0.0 for i in range(3))
    out["per_doc"] = {f"{k}_f": [float(x[2]) for x in v] for k, v in per.items()}
    return out

_BERT = {}

def bertscore(preds, refs, model_type=None, lang="en", batch_size=32, device=None):
    try:
        from bert_score import BERTScorer
    except ImportError:
        return None
    key = (model_type, lang, str(device))
    if key not in _BERT:
        _BERT[key] = BERTScorer(model_type=model_type, lang=lang if model_type is None else None, device=device)
    P, R, F = _BERT[key].score(list(preds), list(refs), batch_size=batch_size)
    return {"bertscore_p": float(P.mean()), "bertscore_r": float(R.mean()), "bertscore_f": float(F.mean()),
            "per_doc_f": [float(x) for x in F]}

def free_bertscore():
    _BERT.clear()

_SENT_SPLIT = re.compile(r"(?<=[.?!])\s+")

def redundancy(text):
    t = _tokens(text)
    g = list(zip(t, t[1:], t[2:]))
    rep3 = 1 - len(set(g)) / len(g) if g else 0.0
    sents = [s.strip().lower() for s in _SENT_SPLIT.split(text) if len(s.split()) >= 4]
    rep_s = 1 - len(set(sents)) / len(sents) if sents else 0.0
    return rep3, rep_s

def novelty(summary, source, n):
    s = list(_ngrams(_tokens(summary), n))
    if not s:
        return 0.0
    src = set(_ngrams(_tokens(source), n))
    return sum(x not in src for x in s) / len(s)

_FACT_PATTERNS = [
    r"\b(?:section|sec|article|art|rule|order|clause|schedule|regulation)s?\.?\s*\(?\d+[a-z]?\)?(?:\s*\(\d+\))?",
    r"\b[a-z][a-z ,.'&()-]{2,60}?\bact\b,?\s*(?:of\s*)?\d{4}",
    r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
    r"\b\d{1,2}(?:st|nd|rd|th)?\s+(?:january|february|march|april|may|june|july|august|september|october|november|"
    r"december)[,]?\s+\d{4}\b",
    r"\b(?:rs\.?|rupees|inr)\s*[\d,]+(?:\.\d+)?",
    r"\b(?:no\.?|nos\.?)\s*\d+(?:\s*(?:of|/)\s*\d{2,4})?",
    r"\b(?:18|19|20)\d{2}\b",
]
_FACT_RE = re.compile("|".join(f"(?:{p})" for p in _FACT_PATTERNS))

def legal_facts(text):
    out = set()
    for m in _FACT_RE.finditer(text.lower()):
        f = re.sub(r"[^a-z0-9]+", " ", m.group(0)).strip()
        f = re.sub(r"^(sec|sections?|s)\b", "section", f)
        f = re.sub(r"^(art|articles?)\b", "article", f)
        if f:
            out.add(f)
    return out

def fact_scores(pred, source, ref):
    gp, src, rf = legal_facts(pred), legal_facts(source), legal_facts(ref)
    prec = len(gp & src) / len(gp) if gp else float("nan")
    rec = len(gp & rf) / len(rf) if rf else float("nan")
    return prec, rec, len(gp)

def analysis(preds, refs, sources):
    rows = []
    for p, r, s in zip(preds, refs, sources):
        rep3, reps = redundancy(p)
        fp, fr, nf = fact_scores(p, s, r)
        rows.append({"words": len(p.split()), "ref_words": len(r.split()), "rep_3gram": rep3, "rep_sent": reps,
                     "novel_1gram": novelty(p, s, 1), "novel_2gram": novelty(p, s, 2), "novel_3gram": novelty(p, s, 3),
                     "fact_precision": fp, "fact_recall": fr, "facts_per_summary": nf})
    out = {}
    for k in rows[0] if rows else []:
        v = np.array([x[k] for x in rows], dtype=float)
        out[k] = float(np.nanmean(v)) if np.isfinite(v).any() else float("nan")
    return out

def length_bucket_report(per_doc, ref_lens, edges=(0, 300, 600, 1000, 10 ** 9)):
    per_doc, ref_lens = np.asarray(per_doc), np.asarray(ref_lens)
    out = []
    for a, b in zip(edges, edges[1:]):
        m = (ref_lens >= a) & (ref_lens < b)
        if m.any():
            out.append({"ref_words": f"{a}-{b if b < 10 ** 9 else 'inf'}", "docs": int(m.sum()),
                        "score": round(float(per_doc[m].mean()), 4)})
    return out

In [ ]:
%%writefile {WORK}/p6summ/train.py
"""Training, generation, evaluation, submission and run management for abstractive summarisation.

One run = one config (DEFAULTS + overrides):
  1. Phase-1 checkpoint -> Summarizer (model.py) -> fine-tuning / PEFT method (peft.py) -> RoPE positions for the
     needed sequence length (position interpolation if longer than pretrained).
  2. optional domain-adaptive pretraining (`dapt.steps` > 0): causal LM on windows of the Phase-6 *training*
     documents with the same trainable parameters, before the summarisation fine-tuning (Gururangan et al., 2020).
  3. fine-tuning: teacher-forced token cross-entropy on the reference summary (truncated to `tgt_len` tokens;
     <eot> appended when the whole summary fits). KD runs add the teacher's token distributions
     (model.seq_loss). An "epoch" = one pass over the training documents (or `docs_per_epoch` of them); the dev
     loss (all dev documents) is measured before training and after every epoch and the best weights are kept;
     `max_minutes` stops a run early (evaluating first) so the Kaggle session limit is never hit.
  4. the best weights generate summaries for `eval.gen_docs` dev documents (ROUGE + analysis measures, saved) and
     for every test document -> checked submission.zip.
"""
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch

from . import data as D
from .context import fid_chunks, source_for
from .generate import GEN_DEFAULTS, generate
from .metrics import analysis, rouge_scores
from .model import Summarizer, seq_loss, trainable_state
from .peft import AdaLoRAController, apply_method, count_params, is_new_param
from .tokenizer import BPETokenizer

DEFAULTS = {
    "run_name": "debug",
    "out_dir": "runs",
    "data_dir": "",
    "cache_dir": "",
    "ckpt_root": "",
    "data": {"dev_size": 300, "dev_frac": 0.05, "split_seed": 0},
    "backbone": {"ckpt": "", "keep_layers": None, "grad_ckpt": True, "random_init": False, "rope_scale": "auto"},
    "method": {"name": "full"},
    "context": {"src_len": 1400, "strategy": "headtail", "head_frac": 0.5, "k": 4, "lead": 256, "gap": True},
    "target": {"tgt_len": "auto", "tgt_quantile": 0.9, "tgt_cap": 640},
    "sep_text": "\n\nSUMMARY:\n",
    "kd": {"teacher_run": "", "kd_alpha": 0.5, "kd_T": 2.0},
    "dapt": {"steps": 0, "len": 1024, "batch_size": 4, "grad_accum": 2},
    "adalora": {"target_r": 4, "init_warmup": 0.1, "final_warmup": 0.2, "delta_t": 10, "orth_reg": 0.1},
    "train": {"epochs": 3, "docs_per_epoch": None, "batch_size": 4, "grad_accum": 4, "lr_backbone": 5e-5,
              "lr_new": 5e-4, "weight_decay": 0.01, "warmup_frac": 0.05, "patience": 2, "grad_clip": 1.0,
              "seed": 1337, "max_minutes": None, "eval_batch_size": 8, "log_every": 0.1, "gen_eval_docs": 16},
    "gen": dict(GEN_DEFAULTS, max_new="auto", min_new="auto"),
    "post": {"dedup_sentences": True, "max_words": None},
    "eval": {"gen_docs": 200},
    "predict_test": True,
    "max_docs": None,
    "n_proc": 4,
}

def merge(base, over):
    out = json.loads(json.dumps(base))
    for k, v in (over or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = merge(out[k], v)
        else:
            out[k] = v
    return out

def set_by_path(cfg, dotted, value):
    try:
        value = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        pass
    d = cfg
    ks = dotted.split(".")
    for k in ks[:-1]:
        d = d.setdefault(k, {})
    d[ks[-1]] = value

CKPT_FOLDERS = {"A": "A_llama_decoder", "A2": "A_llama_decoder_v2", "B": "B_encoder_decoder",
                "C": "C_hybrid_prefix_mlm", "D": "D_efficient_swa_mqa"}

def resolve_ckpt(spec, roots=()):
    if not spec.startswith("@"):
        return spec
    folder = CKPT_FOLDERS[spec[1:]]
    for r in [*roots, "/kaggle/input", "/kaggle/working"]:
        r = Path(r)
        if not r.exists():
            continue
        direct = r / folder / "model_final.pt"
        if direct.exists():
            return str(direct)
        for p in r.rglob("model_final.pt"):
            if p.parent.name == folder:
                return str(p)
    raise FileNotFoundError(f"{folder}/model_final.pt not found under {roots} or /kaggle/input")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def tok_path(cfg):
    return str(Path(cfg["backbone"]["ckpt"]).parent / "tokenizer.json")

def cache_dir(cfg):
    return Path(cfg.get("cache_dir") or Path(cfg.get("out_dir", "runs")).resolve().parent / "p6cache")

def load_data(cfg, log=print):
    dc = cfg["data"]
    return D.load_cached(cfg["data_dir"], tok_path(cfg), cache_dir(cfg), cfg.get("max_docs"), cfg.get("n_proc", 4),
                         dc["dev_size"], dc["dev_frac"], dc["split_seed"], log)

def amp_dtype(device):
    if device != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(obj, open(path, "w", encoding="utf-8"), indent=2, default=str)

def resolve_spec(cfg, data, tok):
    if cfg.get("resolved"):
        return cfg["resolved"]
    t = cfg["target"]
    if t["tgt_len"] == "auto":
        lens = np.array([len(x) for x in data["train"]["sum_tokens"]]) + 1
        tgt = int(min(t["tgt_cap"], math.ceil(np.quantile(lens, t["tgt_quantile"]) / 16) * 16))
    else:
        tgt = int(t["tgt_len"])
    sep = tok.encode(cfg["sep_text"])
    g = cfg["gen"]
    max_new = tgt if g["max_new"] == "auto" else int(g["max_new"])
    if g["min_new"] == "auto":
        lens = np.array([len(x) for x in data["train"]["sum_tokens"]])
        min_new = int(min(max_new // 2, np.quantile(lens, 0.1)))
    else:
        min_new = int(g["min_new"])
    return {"tgt_len": tgt, "src_len": int(cfg["context"]["src_len"]), "sep_ids": sep, "max_new": max_new,
            "min_new": min_new, "gap_ids": tok.encode(" ...") if cfg["context"].get("gap", True) else []}

def build_model(cfg, spec, tok):
    bc = cfg["backbone"]
    model = Summarizer(bc["ckpt"], bc.get("keep_layers"), bc.get("grad_ckpt", True), bc.get("random_init", False))
    model.bos_id = tok.bos_id
    apply_method(model, cfg["method"])
    if model.arch == "B":
        train_len = spec["tgt_len"] + 1
        table = max(spec["max_new"], spec["tgt_len"]) + 2
    else:
        train_len = spec["src_len"] + len(spec["sep_ids"]) + spec["tgt_len"]
        table = spec["src_len"] + len(spec["sep_ids"]) + max(spec["max_new"], spec["tgt_len"]) + 2
    model.rope_info = model.set_lengths(train_len, bc.get("rope_scale", "auto"), table)
    info = {"arch": model.arch, "class": model.cls_name, "model_kw": model.kw, "phase1_layers": model.phase1_layers,
            "layers_used": model.n_layers(), "rope": model.rope_info, "train_seq_len": train_len,
            "random_init": bool(bc.get("random_init"))}
    if model.arch == "B":
        info["fid_chunks"] = math.ceil(spec["src_len"] / model.enc_len)
    return model, info

def load_run(run_dir, device="cuda", weights="best.pt"):
    run_dir = Path(run_dir)
    cfg = json.load(open(run_dir / "config.json"))
    tok = BPETokenizer.load(tok_path(cfg))
    model, _ = build_model(cfg, cfg["resolved"], tok)
    sd = torch.load(run_dir / weights, map_location="cpu")
    _, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, unexpected
    return model.to(device), cfg, tok

def make_target(sum_tokens, tgt_len, eot):
    s = [int(x) for x in sum_tokens]
    return s + [eot] if len(s) <= tgt_len - 1 else s[:tgt_len]

def example(split, i, cfg, spec, eot, with_target=True):
    ex = {"src": source_for(split, i, cfg["context"], spec["src_len"], spec["gap_ids"]).astype(np.int64)}
    if with_target:
        ex["tgt"] = make_target(split["sum_tokens"][i], spec["tgt_len"], eot)
    return ex

def collate(exs, arch, spec, tok, enc_len=None):
    pad = tok.pad_id
    if arch == "B":
        chunks = [(b, k, c) for b, ex in enumerate(exs) for k, c in enumerate(fid_chunks(ex["src"], enc_len))]
        S = max(len(c) for _, _, c in chunks)
        src = torch.full((len(chunks), S), pad, dtype=torch.long)
        keep = torch.zeros(len(chunks), S, dtype=torch.bool)
        for r, (_, _, c) in enumerate(chunks):
            src[r, :len(c)] = torch.from_numpy(c)
            keep[r, :len(c)] = True
        T = max(len(ex["tgt"]) for ex in exs)
        dec_in = torch.full((len(exs), T), pad, dtype=torch.long)
        labels = torch.full((len(exs), T), -100, dtype=torch.long)
        for b, ex in enumerate(exs):
            t = ex["tgt"]
            dec_in[b, :len(t)] = torch.tensor([tok.bos_id] + t[:-1])
            labels[b, :len(t)] = torch.tensor(t)
        return {"src": src, "keep": keep, "doc": torch.tensor([b for b, _, _ in chunks]),
                "slot": torch.tensor([k for _, k, _ in chunks]), "n_docs": len(exs),
                "k_max": max(k for _, k, _ in chunks) + 1, "dec_in": dec_in, "labels": labels}
    seqs, labs, pls = [], [], []
    for ex in exs:
        if "lm" in ex:
            w = list(map(int, ex["lm"]))
            seqs.append(w[:-1]); labs.append(w[1:]); pls.append(0)
            continue
        prompt = list(map(int, ex["src"])) + spec["sep_ids"]
        seq = prompt + ex["tgt"]
        seqs.append(seq[:-1])
        labs.append([-100] * (len(prompt) - 1) + ex["tgt"])
        pls.append(len(prompt))
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), pad, dtype=torch.long)
    labels = torch.full((len(seqs), T), -100, dtype=torch.long)
    for b, (s, l) in enumerate(zip(seqs, labs)):
        assert len(s) == len(l)
        ids[b, :len(s)] = torch.tensor(s)
        labels[b, :len(l)] = torch.tensor(l)
    return {"ids": ids, "labels": labels, "prefix_len": torch.tensor(pls, dtype=torch.long)}

def to_dev(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

def gen_input(ex, arch, spec, enc_len=None):
    if arch == "B":
        return [torch.from_numpy(c) for c in fid_chunks(ex["src"], enc_len)]
    return torch.tensor(list(map(int, ex["src"])) + spec["sep_ids"], dtype=torch.long)

@torch.no_grad()
def dev_loss(model, split, idx, cfg, spec, tok, device, bs):
    model.eval()
    dt = amp_dtype(device)
    eot = tok.special_to_id["<|endoftext|>"]
    tot, n = 0.0, 0
    order = sorted(idx, key=lambda i: len(split["tokens"][i]))
    for k in range(0, len(order), bs):
        exs = [example(split, i, cfg, spec, eot) for i in order[k:k + bs]]
        b = to_dev(collate(exs, model.arch, spec, tok, getattr(model, "enc_len", None)), device)
        with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
            lg, lab = model(b)
        tot += float(torch.nn.functional.cross_entropy(lg.float(), lab, reduction="sum"))
        n += int(lab.numel())
    return tot / max(1, n)

def postprocess(text, pcfg):
    t = " ".join(text.replace("�", " ").split())
    if pcfg.get("dedup_sentences", True):
        seen, keep = set(), []
        for s in D.split_sentences(t):
            key = " ".join(s.lower().split())
            if len(key.split()) >= 4 and key in seen:
                continue
            seen.add(key); keep.append(s)
        t = "".join(keep).strip()
    if pcfg.get("max_words"):
        t = " ".join(t.split()[:int(pcfg["max_words"])])
    return t

def summarise(model, split, idx, cfg, spec, tok, device, gcfg=None, ccfg=None, log=None):
    c = dict(cfg) if ccfg is None else merge(cfg, {"context": ccfg})
    eot = tok.special_to_id["<|endoftext|>"]
    inputs = [gen_input(example(split, i, c, spec, eot, with_target=False), model.arch, spec,
                        getattr(model, "enc_len", None)) for i in idx]
    g = merge(cfg["gen"], gcfg or {})
    g["max_new"] = spec["max_new"] if g["max_new"] == "auto" else int(g["max_new"])
    g["min_new"] = spec["min_new"] if g["min_new"] == "auto" else int(g["min_new"])
    toks, speed = generate(model, inputs, g, tok, device, log, amp_dtype(device))
    texts = [postprocess(tok.decode(t), cfg["post"]) for t in toks]
    return texts, [len(t) for t in toks], speed

def score_generations(texts, split, idx):
    refs = [split["summaries"][i] for i in idx]
    srcs = [split["documents"][i] for i in idx]
    r = rouge_scores(texts, refs)
    a = analysis(texts, refs, srcs)
    return {k: v for k, v in r.items() if k != "per_doc"} | a, r["per_doc"]

def check_prediction(obj, raw_ids):
    assert isinstance(obj, list), "prediction.json must be a JSON list"
    got = [r["id"] for r in obj]
    assert len(got) == len(raw_ids), f"{len(got)} predictions for {len(raw_ids)} test documents"
    assert len({json.dumps(g) for g in got}) == len(got), "duplicate ids in the submission"
    assert {json.dumps(g) for g in got} == {json.dumps(g) for g in raw_ids}, "submission ids differ from test.json ids"
    for r in obj:
        assert set(r) == {"id", "summary"}, f"unexpected keys {sorted(r)}"
        assert isinstance(r["summary"], str) and r["summary"].strip(), f"empty summary for {r['id']!r}"

def write_submission(raw_ids, texts, out_dir):
    import zipfile
    obj = [{"id": rid, "summary": t} for rid, t in zip(raw_ids, texts)]
    check_prediction(obj, raw_ids)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pj = out_dir / "prediction.json"
    json.dump(obj, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = out_dir / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
        check_prediction(json.loads(z.read("prediction.json")), raw_ids)
    return zp

def fill_empty(texts, split, idx, n_words=60):
    out, n = [], 0
    for t, i in zip(texts, idx):
        if not t.strip():
            t, n = " ".join(split["documents"][i].split()[:n_words]) or ".", n + 1
        out.append(t)
    return out, n

def make_optimizer(model, tc, device, lr_mult=1.0):
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        new = is_new_param(n)
        lr = (tc["lr_new"] if new else tc["lr_backbone"]) * lr_mult
        wd = tc["weight_decay"] if p.dim() >= 2 else 0.0
        groups.setdefault((new, wd), {"params": [], "lr": lr, "weight_decay": wd})["params"].append(p)
    return torch.optim.AdamW(list(groups.values()), fused=(device == "cuda"))

def length_batches(idx, lens, bs, rng):
    items = list(map(int, idx))
    rng.shuffle(items)
    batches, block = [], bs * 50
    for c in range(0, len(items), block):
        part = sorted(items[c:c + block], key=lambda i: lens[i])
        batches += [part[i:i + bs] for i in range(0, len(part), bs)]
    rng.shuffle(batches)
    return batches

class Teacher:
    def __init__(self, run_dir, device, student_spec):
        self.model, self.cfg, self.tok = load_run(run_dir, device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.spec = self.cfg["resolved"]
        assert self.spec["tgt_len"] == student_spec["tgt_len"], \
            f"teacher tgt_len {self.spec['tgt_len']} != student {student_spec['tgt_len']}: same summary targets needed"
        self.eot = self.tok.special_to_id["<|endoftext|>"]

    @torch.no_grad()
    def logits(self, split, batch_idx, device, student_labels):
        exs = [example(split, i, self.cfg, self.spec, self.eot) for i in batch_idx]
        b = to_dev(collate(exs, self.model.arch, self.spec, self.tok, getattr(self.model, "enc_len", None)), device)
        dt = amp_dtype(device)
        with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
            lg, lab = self.model(b)
        assert torch.equal(lab, student_labels), "teacher and student summary tokens are not aligned"
        return lg

def dapt(model, tr, cfg, spec, tok, device, log, deadline):
    dc, tc = cfg["dapt"], cfg["train"]
    assert model.arch != "B", "DAPT is implemented for the decoder-style models (A, C, D)"
    L = min(dc["len"], spec["src_len"] + len(spec["sep_ids"]) + spec["tgt_len"])
    rng = np.random.default_rng(tc["seed"] + 1)
    long_docs = [i for i, t in enumerate(tr["tokens"]) if len(t) > L + 1]
    opt = make_optimizer(model, tc, device)
    base = [g["lr"] for g in opt.param_groups]
    dt = amp_dtype(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(dt == torch.float16))
    trainable = [p for p in model.parameters() if p.requires_grad]
    model.train()
    t0, run_loss = time.time(), []
    for step in range(dc["steps"]):
        for _ in range(dc["grad_accum"]):
            exs = []
            for i in rng.choice(long_docs, dc["batch_size"]):
                s = int(rng.integers(0, len(tr["tokens"][i]) - L - 1))
                exs.append({"lm": tr["tokens"][i][s:s + L + 1]})
            b = to_dev(collate(exs, model.arch, spec, tok), device)
            with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
                lg, lab = model(b)
            loss = torch.nn.functional.cross_entropy(lg.float(), lab) / dc["grad_accum"]
            scaler.scale(loss).backward()
            run_loss.append(float(loss.detach()) * dc["grad_accum"])
        mult = min(1.0, (step + 1) / max(1, dc["steps"] // 20)) * max(0.0, 1 - step / dc["steps"])
        for g, b0 in zip(opt.param_groups, base):
            g["lr"] = b0 * mult
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, tc["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if (step + 1) % max(1, dc["steps"] // 10) == 0:
            log(f"[dapt] step {step + 1}/{dc['steps']} | LM loss {np.mean(run_loss[-50:]):.4f} | {(time.time() - t0) / 60:.1f} min")
        if deadline and time.time() > deadline:
            log("[dapt] time budget reached"); break
    return {"steps": step + 1, "len": L, "first_loss": float(np.mean(run_loss[:20])), "last_loss": float(np.mean(run_loss[-50:])),
            "minutes": round((time.time() - t0) / 60, 1)}

def fit(model, tr, dev, cfg, spec, tok, device, log, run_dir, teacher=None):
    tc = cfg["train"]
    eot = tok.special_to_id["<|endoftext|>"]
    enc_len = getattr(model, "enc_len", None)
    n = len(tr["ids"])
    dpe = min(n, tc.get("docs_per_epoch") or n)
    rng_np = np.random.default_rng(tc["seed"])
    schedule, stream = [], np.array([], dtype=np.int64)
    for _ in range(tc["epochs"]):
        while len(stream) < dpe:
            stream = np.concatenate([stream, rng_np.permutation(n)])
        schedule.append(stream[:dpe]); stream = stream[dpe:]
    lens = [min(len(t), spec["src_len"]) + len(s) for t, s in zip(tr["tokens"], tr["sum_tokens"])]
    opt = make_optimizer(model, tc, device)
    base_lrs = [g["lr"] for g in opt.param_groups]
    steps_per_epoch = math.ceil(math.ceil(dpe / tc["batch_size"]) / tc["grad_accum"])
    total = steps_per_epoch * len(schedule)
    warm = max(1, int(tc["warmup_frac"] * total))
    ada = None
    if cfg["method"]["name"] == "adalora":
        a = cfg["adalora"]
        ada = AdaLoRAController(model, total, a["target_r"], a["init_warmup"], a["final_warmup"], a["delta_t"],
                                orth_reg=a["orth_reg"])
        log(f"[adalora] {len(ada.mods)} modules | rank budget {ada.b0} -> {ada.bT} over {total} steps")
    dt = amp_dtype(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(dt == torch.float16))
    trainable = [p for p in model.parameters() if p.requires_grad]
    dev_idx = list(range(len(dev["ids"])))
    gen_idx = dev_idx[:tc.get("gen_eval_docs", 0)]
    log(f"[train] {len(schedule)} epochs x {dpe} documents | {steps_per_epoch} steps/epoch | {total} steps | "
        f"batch {tc['batch_size']} x accum {tc['grad_accum']} | amp {dt} | time limit {tc.get('max_minutes')} min")
    hist = []
    l0 = dev_loss(model, dev, dev_idx, cfg, spec, tok, device, tc["eval_batch_size"])
    hist.append({"epoch": 0, "dev_loss": round(l0, 4), "dev_ppl": round(math.exp(min(l0, 30)), 2)})
    log(f"[epoch] epoch 0 (before fine-tuning) | dev_loss {l0:.4f} | dev_ppl {math.exp(min(l0, 30)):.2f}")
    best, best_epoch, bad, step = float("inf"), 0, 0, 0
    rng = random.Random(tc["seed"])
    kd = cfg["kd"]
    t0 = time.time()
    limit = tc.get("max_minutes")
    stop = False
    for ep, idx in enumerate(schedule, 1):
        model.train()
        tot, parts, nbat, ntok = 0.0, {}, 0, 0
        batches = length_batches(idx, lens, tc["batch_size"], rng)
        every = max(1, int(len(batches) * tc.get("log_every", 0.1)))
        for bi, bidx in enumerate(batches):
            exs = [example(tr, i, cfg, spec, eot) for i in bidx]
            b = to_dev(collate(exs, model.arch, spec, tok, enc_len), device)
            with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
                lg, lab = model(b)
            tl = None if teacher is None else teacher.logits(tr, bidx, device, lab)
            loss, pr = seq_loss(lg, lab, tl, kd["kd_alpha"], kd["kd_T"])
            if ada is not None:
                o = ada.orth_loss()
                loss = loss + o
                pr["orth"] = float(o.detach())
            scaler.scale(loss / tc["grad_accum"]).backward()
            tot += float(loss.detach()); nbat += 1; ntok += int(lab.numel())
            for k, v in pr.items():
                parts[k] = parts.get(k, 0.0) + v
            if nbat % tc["grad_accum"] == 0 or bi == len(batches) - 1:
                lr_mult = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))
                for g, b0 in zip(opt.param_groups, base_lrs):
                    g["lr"] = b0 * lr_mult
                scaler.unscale_(opt)
                if ada is not None:
                    ada.update(step)
                torch.nn.utils.clip_grad_norm_(trainable, tc["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
            if (bi + 1) % every == 0:
                el = (time.time() - t0) / 60
                done = (ep - 1) + (bi + 1) / len(batches)
                log(f"[progress] epoch {ep} {100 * (bi + 1) / len(batches):3.0f}% | loss {tot / nbat:.4f} | "
                    + " ".join(f"{k} {v / nbat:.4f}" for k, v in parts.items())
                    + f" | {ntok / max(1e-6, el * 60):.0f} summary tok/s | {el:.1f} min | "
                    f"ETA all epochs {el / done * (len(schedule) - done):.1f} min")
            if limit and (time.time() - t0) / 60 > limit:
                log(f"[train] time limit {limit} min reached in epoch {ep} ({100 * (bi + 1) / len(batches):.0f}%): "
                    "evaluating and stopping")
                stop = True
                break
        rec = {"epoch": ep, "train_loss": round(tot / max(nbat, 1), 4), "minutes": round((time.time() - t0) / 60, 2)}
        rec.update({f"train_{k}": round(v / max(nbat, 1), 4) for k, v in parts.items()})
        dl = dev_loss(model, dev, dev_idx, cfg, spec, tok, device, tc["eval_batch_size"])
        rec.update(dev_loss=round(dl, 4), dev_ppl=round(math.exp(min(dl, 30)), 2))
        if ada is not None:
            rec["adalora_budget"] = ada.budget
        if gen_idx:
            g = {"max_new": min(256, spec["max_new"]), "min_new": min(spec["min_new"], 64), "method": "greedy"}
            texts, _, _ = summarise(model, dev, gen_idx, cfg, spec, tok, device, g)
            r = rouge_scores(texts, [dev["summaries"][i] for i in gen_idx])
            rec.update(quick_rouge1=round(r["rouge1_f"], 4), quick_rougeL=round(r["rougeL_f"], 4))
            rec["sample"] = texts[0][:300]
        if dl < best:
            best, best_epoch, bad = dl, ep, 0
            torch.save(trainable_state(model), run_dir / "best.pt"); rec["saved"] = True
        else:
            bad += 1
        hist.append(rec)
        log("[epoch] " + " | ".join(f"{k} {v}" for k, v in rec.items() if k != "sample"))
        if "sample" in rec:
            log(f"[epoch] sample: {rec['sample']}")
        save_json(hist, run_dir / "history.json")
        if stop:
            break
        if bad >= tc["patience"]:
            log(f"[train] early stop: no dev improvement for {tc['patience']} epochs")
            break
    return hist, best, best_epoch, (None if ada is None else ada.ranks())

def run(cfg, data=None, device=None, log=print):
    cfg = merge(DEFAULTS, cfg)
    roots = [cfg["ckpt_root"]] if cfg.get("ckpt_root") else []
    cfg["backbone"]["ckpt"] = resolve_ckpt(cfg["backbone"]["ckpt"], roots)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    seed_all(cfg["train"]["seed"])
    run_dir = Path(cfg["out_dir"]) / cfg["run_name"]
    run_dir.mkdir(parents=True, exist_ok=True)
    tok = BPETokenizer.load(tok_path(cfg))
    data = data or load_data(cfg, log)
    tr, dev, te = data["train"], data["dev"], data["test"]
    spec = resolve_spec(cfg, data, tok)
    cfg["resolved"] = spec
    save_json(cfg, run_dir / "config.json")
    log(f"[data] train {len(tr['ids'])} | dev {len(dev['ids'])} (held out) | test {len(te['ids'])} documents | "
        f"src {spec['src_len']} tokens ({cfg['context']['strategy']}) | summary <= {spec['tgt_len']} tokens | "
        f"generation {spec['min_new']}..{spec['max_new']} tokens")

    teacher = None
    if cfg["kd"].get("teacher_run"):
        teacher = Teacher(cfg["kd"]["teacher_run"], device, spec)
        log(f"[kd] teacher {Path(cfg['kd']['teacher_run']).name} ({teacher.model.arch}, "
            f"{count_params(teacher.model)['total'] / 1e6:.1f}M) | alpha {cfg['kd']['kd_alpha']} | T {cfg['kd']['kd_T']}")

    model, info = build_model(cfg, spec, tok)
    model.to(device)
    pc = count_params(model)
    info.update(params=pc)
    save_json(info, run_dir / "model_info.json")
    log(f"[model] arch {info['arch']} ({info['class']}) | layers {info['layers_used']}/{info['phase1_layers']} | "
        f"method {cfg['method']['name']} | positions {info['rope']} | train sequence {info['train_seq_len']}"
        + (f" | FiD {info['fid_chunks']} x {model.enc_len}" if model.arch == "B" else ""))
    log(f"[model] params total {pc['total'] / 1e6:.2f}M | trainable {pc['trainable'] / 1e6:.3f}M ({pc['trainable_pct']}%)"
        + (f" | 4-bit NF4 weights {pc['nf4_weights'] / 1e6:.2f}M" if pc.get("nf4_weights") else ""))
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    dapt_info = None
    if cfg["dapt"].get("steps"):
        lim = cfg["train"].get("max_minutes")
        dapt_info = dapt(model, tr, cfg, spec, tok, device, log, (t_start + 60 * lim / 3) if lim else None)
        if lim:
            cfg["train"]["max_minutes"] = max(5, lim - dapt_info["minutes"])
    hist, best, best_epoch, ada_ranks = fit(model, tr, dev, cfg, spec, tok, device, log, run_dir, teacher)
    train_min = (time.time() - t_start) / 60
    peak_mem = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else None
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()

    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device), strict=False)
    gidx = list(range(min(cfg["eval"]["gen_docs"], len(dev["ids"]))))
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    texts, ntok, speed = summarise(model, dev, gidx, cfg, spec, tok, device, log=log)
    gen_mem = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else None
    texts, n_empty_dev = fill_empty(texts, dev, gidx)
    scores, per_doc = score_generations(texts, dev, gidx)
    save_json([{"id": dev["ids"][i], "reference": dev["summaries"][i], "generated": t, "new_tokens": n,
                "doc_tokens": int(len(dev["tokens"][i]))} | {k: v[j] for k, v in per_doc.items()}
               for j, (i, t, n) in enumerate(zip(gidx, texts, ntok))], run_dir / "dev_generations.json")
    save_json(scores | {"generation": speed, "n_docs": len(gidx), "empty_replaced": n_empty_dev}, run_dir / "dev_metrics.json")
    log(f"[dev] {len(gidx)} documents | ROUGE-1 {scores['rouge1_f']:.4f} | ROUGE-2 {scores['rouge2_f']:.4f} | "
        f"ROUGE-L {scores['rougeL_f']:.4f} | words {scores['words']:.0f} (ref {scores['ref_words']:.0f}) | "
        f"rep-3gram {scores['rep_3gram']:.3f} | fact precision {scores['fact_precision']:.3f} | "
        f"{speed['docs_per_sec']} docs/s")
    summary = {"run": cfg["run_name"], "arch": info["arch"], "backbone": Path(cfg["backbone"]["ckpt"]).parent.name,
               "method": cfg["method"], "context": cfg["context"], "resolved": spec, "rope": info["rope"],
               "layers": info["layers_used"], "params_total": pc["total"], "params_trainable": pc["trainable"],
               "trainable_pct": pc["trainable_pct"], "nf4_weights": pc.get("nf4_weights", 0),
               "train_minutes": round(train_min, 1), "peak_train_mem_gb": peak_mem, "peak_gen_mem_gb": gen_mem,
               "best_epoch": best_epoch, "epochs_run": len(hist) - 1, "best_dev_loss": best,
               "dev_loss_before": hist[0]["dev_loss"], "kd_teacher": cfg["kd"].get("teacher_run") or None,
               "dapt": dapt_info, "adalora_final_ranks": ada_ranks, "gen": cfg["gen"],
               "dev": {k: scores[k] for k in ("rouge1_f", "rouge2_f", "rougeL_f", "words", "ref_words", "rep_3gram",
                                             "rep_sent", "novel_3gram", "fact_precision", "fact_recall")},
               "gen_docs_per_sec": speed["docs_per_sec"], "gen_tokens_per_sec": speed["tokens_per_sec"]}

    if cfg.get("predict_test", True):
        tidx = list(range(len(te["ids"])))
        tt, tn, tspeed = summarise(model, te, tidx, cfg, spec, tok, device, log=log)
        tt, n_empty = fill_empty(tt, te, tidx)
        save_json([{"id": te["ids"][i], "generated": t, "new_tokens": n} for i, t, n in zip(tidx, tt, tn)],
                  run_dir / "test_generations.json")
        zp = write_submission(te["raw_ids"], tt, run_dir / "submission")
        summary.update(submission=str(zp), test_words=round(float(np.mean([len(t.split()) for t in tt])), 1),
                       test_empty_replaced=n_empty, test_gen_seconds=tspeed["seconds"])
        log(f"[submit] {zp} | {len(tt)} documents | mean {summary['test_words']} words | "
            f"empty generations replaced: {n_empty}")
    save_json(summary, run_dir / "summary.json")
    return summary, model

In [ ]:
%%writefile {WORK}/p6summ/run.py
"""CLI + analyses.

    python -m p6summ.run --config job.json                          # train one experiment
    python -m p6summ.run --eda --config job.json                    # data format + statistics only
    python -m p6summ.run --mbr runs/R1 runs/R2 runs/T1 --out runs/ENS_MBR   # consensus selection between runs
"""
import argparse
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch

from . import data as D
from .context import select_source
from .metrics import analysis, bertscore, rouge_fallback, rouge_scores
from .tokenizer import BPETokenizer
from .train import (DEFAULTS, load_data, load_run, merge, save_json, score_generations, set_by_path, summarise,
                    write_submission)

def coverage_analysis(data, tok, budgets=(512, 1024, 1536, 3072), strategies=("head", "tail", "headtail", "uniform",
                                                                                 "salient"), max_docs=200, lead=256):
    dev = data["dev"]
    gap = tok.encode(" ...")
    n = min(max_docs, len(dev["ids"]))
    rows = []
    full = [set(D.content_words(dev["documents"][i])) for i in range(n)]
    refs = [set(D.content_words(dev["summaries"][i])) for i in range(n)]
    cov_full = np.mean([len(r & f) / max(1, len(r)) for r, f in zip(refs, full)])
    for L in budgets:
        row = {"budget_tokens": L, "docs_longer_than_budget": round(float(np.mean([len(dev["tokens"][i]) > L
                                                                                    for i in range(n)])), 3)}
        for st in strategies:
            c = []
            for i in range(n):
                sel = select_source(dev["tokens"][i], L, st, dev["sent_end"][i], dev["sent_score"][i], 0.5, 4, lead, gap)
                w = set(D.content_words(tok.decode(sel)))
                c.append(len(refs[i] & w) / max(1, len(refs[i])))
            row[st] = round(float(np.mean(c)), 4)
        rows.append(row)
    return {"rows": rows, "whole_document": round(float(cov_full), 4), "docs": n}

def _pair_sim(a, b):
    s = rouge_fallback(a, b, lcs=False)
    return (s["rouge1"][2] + s["rouge2"][2]) / 2

def mbr_select(cands):
    if len(cands) == 1:
        return 0, [1.0]
    m = np.zeros((len(cands), len(cands)))
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            m[i, j] = m[j, i] = _pair_sim(cands[i], cands[j])
    mean = m.sum(1) / (len(cands) - 1)
    return int(mean.argmax()), mean.tolist()

def mbr_ensemble(run_dirs, out_dir, data):
    run_dirs = [Path(r) for r in run_dirs]
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    dev, te = data["dev"], data["test"]
    gens = {r.name: json.load(open(r / "dev_generations.json")) for r in run_dirs}
    common = set.intersection(*[{g["id"] for g in v} for v in gens.values()])
    pos = {i: k for k, i in enumerate(dev["ids"])}
    ids = [g["id"] for g in gens[run_dirs[0].name] if g["id"] in common]
    by = {n: {g["id"]: g["generated"] for g in v} for n, v in gens.items()}
    chosen, picks = [], Counter()
    for i in ids:
        names = list(by)
        k, _ = mbr_select([by[n][i] for n in names])
        chosen.append(by[names[k]][i]); picks[names[k]] += 1
    idx = [pos[i] for i in ids]
    scores, per_doc = score_generations(chosen, dev, idx)
    save_json([{"id": dev["ids"][j], "reference": dev["summaries"][j], "generated": t} | {k: v[q] for k, v in per_doc.items()}
               for q, (j, t) in enumerate(zip(idx, chosen))], out_dir / "dev_generations.json")
    tg = {r.name: {g["id"]: g["generated"] for g in json.load(open(r / "test_generations.json"))} for r in run_dirs}
    tsel, tpicks = [], Counter()
    for i in te["ids"]:
        names = list(tg)
        k, _ = mbr_select([tg[n][i] for n in names])
        tsel.append(tg[names[k]][i]); tpicks[names[k]] += 1
    zp = write_submission(te["raw_ids"], tsel, out_dir / "submission")
    s = {"run": out_dir.name, "members": [r.name for r in run_dirs], "dev": scores, "dev_picks": dict(picks),
         "test_picks": dict(tpicks), "submission": str(zp)}
    save_json(s, out_dir / "summary.json")
    return s

def length_sweep(run_dir, data, word_limits=(100, 200, 300, 400, 600, 800), bert=None):
    gens = json.load(open(Path(run_dir) / "dev_generations.json"))
    refs = [g["reference"] for g in gens]
    rows = []
    for L in list(word_limits) + [None]:
        cut = [" ".join(g["generated"].split()[:L]) if L else g["generated"] for g in gens]
        r = rouge_scores(cut, refs)
        row = {"max_words": L or "full", "mean_words": round(float(np.mean([len(c.split()) for c in cut])), 1),
               "rouge1_f": round(r["rouge1_f"], 4), "rouge1_r": round(r["rouge1_r"], 4), "rougeL_f": round(r["rougeL_f"], 4)}
        if bert is not None:
            b = bertscore(cut, refs, **bert)
            if b:
                row["bertscore_f"] = round(b["bertscore_f"], 4)
        rows.append(row)
    return rows

def generation_variants(run_dir, data, variants, n_docs=100, device="cuda", bert=None, log=print):
    model, cfg, tok = load_run(run_dir, device)
    dev = data["dev"]
    idx = list(range(min(n_docs, len(dev["ids"]))))
    spec = cfg["resolved"]
    rows = []
    for name, v in variants.items():
        texts, ntok, speed = summarise(model, dev, idx, cfg, spec, tok, device, v.get("gen"), v.get("context"))
        texts = [t if t.strip() else "." for t in texts]
        sc, _ = score_generations(texts, dev, idx)
        row = {"variant": name, "rouge1_f": round(sc["rouge1_f"], 4), "rouge2_f": round(sc["rouge2_f"], 4),
               "rougeL_f": round(sc["rougeL_f"], 4), "words": round(sc["words"], 1), "rep_3gram": round(sc["rep_3gram"], 4),
               "novel_3gram": round(sc["novel_3gram"], 4), "fact_precision": round(sc["fact_precision"], 4),
               "fact_recall": round(sc["fact_recall"], 4), "docs_per_s": speed["docs_per_sec"]}
        if bert is not None:
            b = bertscore(texts, [dev["summaries"][i] for i in idx], **bert)
            if b:
                row["bertscore_f"] = round(b["bertscore_f"], 4)
        rows.append(row)
        log(f"[variants] {name}: " + " | ".join(f"{k} {v}" for k, v in row.items() if k != "variant"))
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return rows

def regenerate(run_dir, data, out_dir, gen=None, post=None, device="cuda", log=print):
    from .train import fill_empty
    model, cfg, tok = load_run(run_dir, device)
    cfg = merge(cfg, {"gen": gen or {}, "post": post or {}})
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    spec = cfg["resolved"]
    dev, te = data["dev"], data["test"]
    gidx = list(range(min(cfg["eval"]["gen_docs"], len(dev["ids"]))))
    texts, ntok, speed = summarise(model, dev, gidx, cfg, spec, tok, device, log=log)
    texts, _ = fill_empty(texts, dev, gidx)
    scores, per_doc = score_generations(texts, dev, gidx)
    save_json([{"id": dev["ids"][i], "reference": dev["summaries"][i], "generated": t, "new_tokens": n} |
               {k: v[j] for k, v in per_doc.items()} for j, (i, t, n) in enumerate(zip(gidx, texts, ntok))],
              out_dir / "dev_generations.json")
    tidx = list(range(len(te["ids"])))
    tt, tn, _ = summarise(model, te, tidx, cfg, spec, tok, device, log=log)
    tt, n_empty = fill_empty(tt, te, tidx)
    save_json([{"id": te["ids"][i], "generated": t, "new_tokens": n} for i, t, n in zip(tidx, tt, tn)],
              out_dir / "test_generations.json")
    zp = write_submission(te["raw_ids"], tt, out_dir / "submission")
    s = {"run": out_dir.name, "from_run": str(run_dir), "gen": cfg["gen"], "post": cfg["post"], "dev": scores,
         "generation": speed, "submission": str(zp), "test_empty_replaced": n_empty}
    save_json(cfg, out_dir / "config.json")
    save_json(s, out_dir / "summary.json")
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return s

def bertscore_run(run_dir, bert_kw):
    run_dir = Path(run_dir)
    out = run_dir / "dev_bertscore.json"
    if out.exists():
        return json.load(open(out))
    gens = json.load(open(run_dir / "dev_generations.json"))
    b = bertscore([g["generated"] for g in gens], [g["reference"] for g in gens], **bert_kw)
    if b is None:
        return None
    for g, f in zip(gens, b["per_doc_f"]):
        g["bertscore_f"] = f
    save_json(gens, run_dir / "dev_generations.json")
    res = {k: v for k, v in b.items() if k != "per_doc_f"}
    save_json(res, out)
    return res

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--set", nargs="*", default=[], help="dotted overrides, e.g. train.epochs=2")
    ap.add_argument("--eda", action="store_true")
    ap.add_argument("--mbr", nargs="*", default=None)
    ap.add_argument("--out", default="runs/ENS_MBR")
    a = ap.parse_args()
    cfg = json.load(open(a.config)) if a.config else {}
    for kv in a.set:
        k, v = kv.split("=", 1)
        set_by_path(cfg, k, v)
    from .train import run, resolve_ckpt
    if a.eda or a.mbr:
        c = merge(DEFAULTS, cfg)
        c["backbone"]["ckpt"] = resolve_ckpt(c["backbone"]["ckpt"], [c["ckpt_root"]] if c.get("ckpt_root") else [])
        data = load_data(c)
        if a.eda:
            D.describe_raw(c["data_dir"])
            print(json.dumps(D.stats(data), indent=1))
            tok = BPETokenizer.load(str(Path(c["backbone"]["ckpt"]).parent / "tokenizer.json"))
            print(json.dumps(coverage_analysis(data, tok), indent=1))
        if a.mbr:
            print(json.dumps(mbr_ensemble(a.mbr, a.out, data)["dev"], indent=1))
        return
    run(cfg)

if __name__ == "__main__":
    main()

In [ ]:
%%writefile {WORK}/tests/test_pipeline.py
"""Correctness checks for the Phase-6 pipeline (run before trusting any number).

    python tests/test_pipeline.py [--ckpt_dir <folder with the Phase-1 run folders>]
"""
import argparse
import json
import os
import sys
import tempfile

import numpy as np
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from p6summ import peft
from p6summ.context import _cost, fid_chunks, select_source, select_spans
from p6summ.data import _records, norm_doc, sentence_scores, split_sentences
from p6summ.generate import GEN_DEFAULTS, generate, process_logits
from p6summ.metrics import legal_facts, redundancy, rouge_fallback, rouge_scores
from p6summ.model import Summarizer, load_phase1, map_state, seq_loss
from p6summ.phase1_models import MODEL_CLASSES
from p6summ.train import check_prediction, collate, make_target, postprocess, write_submission

V = 300
TINY = {"LlamaLM": dict(vocab_size=V, block_size=64, n_layer=2, n_head=4, n_kv_head=2, d_model=32),
        "EncDecLM": dict(vocab_size=V, enc_block=32, dec_block=32, n_enc_layer=2, n_dec_layer=2, n_head=4, d_model=32),
        "HybridLM": dict(vocab_size=V, block_size=64, n_layer=2, n_head=4, d_model=32),
        "EfficientLM": dict(vocab_size=V, block_size=64, window=8, n_layer=2, n_head=4, n_kv_head=1, d_model=32)}
TMP = tempfile.mkdtemp()

class FakeTok:
    special_to_id = {"<|endoftext|>": V - 5, "<|pad|>": V - 4, "<|bos|>": V - 3, "<|sep|>": V - 2, "<|mask|>": V - 1}
    pad_id, bos_id, mask_id = V - 4, V - 3, V - 1

    def decode(self, ids):
        return " ".join(f"w{i}" for i in ids)

def tiny_ckpt(cls, seed=0):
    torch.manual_seed(seed)
    m = MODEL_CLASSES[cls](**TINY[cls])
    if cls == "HybridLM":
        torch.nn.init.normal_(m.seg_emb.weight, std=0.02)
    p = os.path.join(TMP, f"{cls}_{seed}.pt")
    torch.save({"model_class": cls, "model_kw": TINY[cls], "model": m.state_dict()}, p)
    return p, m.eval()

def test_sentences_and_context():
    rng = np.random.default_rng(0)
    for text in ["The appellant filed a suit. It was dismissed; the High Court allowed it. Held: appeal fails.",
                 "No punctuation at all " * 200, "A. B. C. 1. 2. (a) x", ""]:
        assert "".join(split_sentences(text)) == text
    s = split_sentences("Section 302 IPC applies. The accused was convicted. Appeal dismissed.")
    assert len(s) == 3 and len(sentence_scores(s)) == 3
    for _ in range(3000):
        n = int(rng.integers(1, 5000)); L = int(rng.choice([16, 64, 512, 1400])); gl = int(rng.integers(0, 3))
        m = int(rng.integers(1, 80))
        ends = np.unique(np.sort(np.concatenate([rng.integers(1, n + 1, m), [n]]))).astype(np.int32)
        score = rng.random(len(ends)).astype(np.float32)
        for st in ("head", "tail", "headtail", "uniform", "salient"):
            sp = select_spans(n, L, st, ends, score, float(rng.choice([0.3, 0.5])), int(rng.integers(1, 6)),
                              int(rng.choice([0, 64, 256])), gl)
            assert all(0 <= a < b <= n for a, b in sp), (n, L, st, sp)
            assert all(sp[i][1] < sp[i + 1][0] for i in range(len(sp) - 1)), (st, sp)
            assert _cost(sp, gl) <= max(L, 0) or n <= L, (n, L, st, sp, gl)
            if n <= L:
                assert sp == [(0, n)]
    toks = np.arange(1000, dtype=np.uint16)
    out = select_source(toks, 100, "headtail", gap_ids=[7, 7])
    assert len(out) == 100 and list(out[:49]) == list(range(49)) and list(out[49:51]) == [7, 7] and out[-1] == 999
    assert [len(c) for c in fid_chunks(np.arange(1700), 768)] == [768, 768, 164]
    print("ok  sentences + context selection (spans in budget, ordered, gap markers)")

def test_loader():
    recs = _records({"d1": {"document": ["para one.", "para two."], "summary": "s"}, "d2": {"id": "d2", "text": "x",
                                                                                          "headnote": "h"}})
    a, b = norm_doc(recs[0], "t", True), norm_doc(recs[1], "t", True)
    assert a["id"] == "d1" and a["document"] == "para one. para two." and b["summary"] == "h"
    try:
        norm_doc({"id": 1, "document": "x"}, "t", True); raise AssertionError("missing summary accepted")
    except ValueError:
        pass
    print("ok  loader (dict / list formats, key aliases, missing summary rejected)")

def test_targets_and_collate():
    tok = FakeTok()
    eot = tok.special_to_id["<|endoftext|>"]
    assert make_target([1, 2, 3], 5, eot) == [1, 2, 3, eot] and make_target([1, 2, 3, 4, 5, 6], 5, eot) == [1, 2, 3, 4, 5]
    assert make_target([1, 2, 3, 4], 5, eot) == [1, 2, 3, 4, eot]
    spec = {"sep_ids": [9, 9]}
    exs = [{"src": np.array([5, 6, 7]), "tgt": [1, 2, eot]}, {"src": np.array([5]), "tgt": [3, 4, 8, 8, eot]}]
    b = collate(exs, "A", spec, tok)
    for r in range(2):
        ids, lab = b["ids"][r].tolist(), b["labels"][r].tolist()
        full = list(exs[r]["src"]) + [9, 9] + exs[r]["tgt"]
        for p, l in enumerate(lab):
            if l != -100:
                assert full[p + 1] == l
        assert [l for l in lab if l != -100] == exs[r]["tgt"]
        assert b["prefix_len"][r] == len(exs[r]["src"]) + 2
    bb = collate([{"src": np.arange(70), "tgt": [1, 2, eot]}, {"src": np.arange(10), "tgt": [4, eot]}], "B", spec, tok, 32)
    assert bb["src"].shape == (4, 32) and bb["k_max"] == 3 and bb["doc"].tolist() == [0, 0, 0, 1]
    assert bb["dec_in"][0].tolist() == [tok.bos_id, 1, 2] and bb["labels"][1].tolist() == [4, eot, -100]
    print("ok  targets (<eot> only when the summary fits) + collate alignment (decoder-style and FiD)")

def _build(cls, method=None, seed=0):
    p, ref = tiny_ckpt(cls, seed)
    s = Summarizer(p).eval()
    s.bos_id = FakeTok.bos_id
    if method:
        peft.apply_method(s, method)
    s.set_lengths(32 if cls == "EncDecLM" else 64)
    return s, ref

def test_equivalence():
    for cls in TINY:
        s, m = _build(cls)
        torch.manual_seed(1)
        if cls == "EncDecLM":
            src = torch.randint(0, 290, (2, 30)); keep = torch.ones(2, 30, dtype=torch.bool); keep[1, 20:] = False
            dec = torch.randint(0, 290, (2, 20))
            ref, _ = m(src, keep, dec)
            got, _ = s.forward_encdec(src, keep, torch.arange(2), torch.zeros(2, dtype=torch.long), 2, 1, dec,
                                      torch.zeros(2, 20, dtype=torch.long))
            assert (ref.reshape(-1, V) - got).abs().max() < 1e-4
            st = s.prefill([[src[0]], [src[1, :20]]])
            lg, seq = [st["logits"]], [torch.full((2,), FakeTok.bos_id)]
            for _ in range(12):
                t = torch.randint(0, 290, (2,)); seq.append(t); lg.append(s.step(st, t))
            dec_in = torch.stack(seq, 1)
            for b, L in ((0, 30), (1, 20)):
                r, _ = m(src[b:b + 1, :L], None, dec_in[b:b + 1])
                assert (r[0] - torch.stack([x[b] for x in lg])).abs().max() < 1e-4, cls
            continue
        ids = torch.randint(0, 290, (2, 48))
        pl = torch.tensor([20, 31]) if cls == "HybridLM" else None
        ref, _ = m(ids, pl) if cls == "HybridLM" else m(ids)
        got, _ = s.forward_dec(ids, torch.zeros(2, 48, dtype=torch.long), pl)
        assert (ref.reshape(-1, V) - got).abs().max() < 1e-4, cls
        P = [ids[0, :5], ids[1, :31]]
        st = s.prefill(P)
        lg, toks = [st["logits"]], []
        for _ in range(14):
            t = torch.randint(0, 290, (2,)); toks.append(t); lg.append(s.step(st, t))
        for b in range(2):
            full = torch.cat([P[b], torch.stack(toks, 1)[b]])[None]
            r, _ = m(full, torch.tensor([len(P[b])])) if cls == "HybridLM" else m(full)
            assert (r[0, len(P[b]) - 1:] - torch.stack([x[b] for x in lg])).abs().max() < 1e-4, (cls, b)
    print("ok  Summarizer == Phase-1 forward (A, B, C, D) and KV-cached batched generation == full forward")

def _gen_vs_forward(s, P, n=8):
    st = s.prefill(P)
    lg, toks = [st["logits"]], []
    for _ in range(n):
        t = torch.randint(0, 290, (len(P),)); toks.append(t); lg.append(s.step(st, t))
    for b in range(len(P)):
        full = torch.cat([P[b], torch.stack(toks, 1)[b]])
        lab = torch.zeros(1, len(full), dtype=torch.long)
        lab[0, :len(P[b]) - 1] = -100
        got, _ = s.forward_dec(full[None], lab, torch.tensor([len(P[b])]))
        assert (got - torch.stack([x[b] for x in lg])[:got.size(0)]).abs().max() < 1e-4

def test_peft():
    torch.manual_seed(0)
    ident = [{"name": "lora", "r": 4}, {"name": "lorma", "r": 4}, {"name": "lorma", "variant": "pi", "r": 4},
             {"name": "houlsby", "bottleneck": 8}, {"name": "pfeiffer", "bottleneck": 8}, {"name": "ia3"},
             {"name": "bitfit"}, {"name": "adalora", "r": 4}]
    for cls in TINY:
        base, _ = _build(cls)
        ids = torch.randint(0, 290, (2, 30))
        if cls == "EncDecLM":
            args = (ids, torch.ones(2, 30, dtype=torch.bool), torch.arange(2), torch.zeros(2, dtype=torch.long), 2, 1,
                    ids[:, :12], torch.zeros(2, 12, dtype=torch.long))
            f = lambda m: m.forward_encdec(*args)[0]
        else:
            f = lambda m: m.forward_dec(ids, torch.zeros(2, 30, dtype=torch.long), torch.tensor([10, 12]))[0]
        y0 = f(base)
        for mc in ident + [{"name": "qlora", "r": 4}, {"name": "partial", "train_last_k": 1}]:
            m, _ = _build(cls, mc)
            n_tr = peft.count_params(m)["trainable"]
            assert n_tr > 0, (cls, mc)
            if mc["name"] != "partial":
                assert all(peft.is_new_param(n) for n, p in m.named_parameters() if p.requires_grad
                           and not ("norm" in n and mc["name"] == "bitfit")), (cls, mc)
            d = (f(m) - y0).abs().max().item()
            assert d < (0.5 if mc["name"] == "qlora" else 1e-4), (cls, mc, d)
            out = f(m); out.float().sum().backward()
            assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in m.parameters() if p.requires_grad), (cls, mc)
    for cls in ("LlamaLM", "HybridLM"):
        for mc in ({"name": "prefix", "n_prefix": 3}, {"name": "prompt", "n_prompt": 4}):
            m, _ = _build(cls, mc)
            with torch.no_grad():
                for p in m.parameters():
                    if p.requires_grad:
                        p.add_(torch.randn_like(p) * 0.1)
            _gen_vs_forward(m, [torch.randint(0, 290, (7,)), torch.randint(0, 290, (12,))])
    m, _ = _build("EfficientLM", {"name": "prompt", "n_prompt": 4})
    _gen_vs_forward(m, [torch.randint(0, 290, (5,)), torch.randint(0, 290, (12,))])
    print("ok  PEFT methods on A/B/C/D: identity at init, only new params trainable, gradients flow; "
          "prefix/prompt generation == forward")

def test_adalora_controller():
    m, _ = _build("LlamaLM", {"name": "adalora", "r": 6})
    ctl = peft.AdaLoRAController(m, total_steps=100, target_r=2, init_warmup=0.1, final_warmup=0.2, delta_t=5)
    b = [ctl.schedule(t) for t in range(100)]
    assert b[0] == ctl.b0 and b[-1] == ctl.bT and all(x >= y for x, y in zip(b, b[1:]))
    opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=0.01)
    ids = torch.randint(0, 290, (2, 20))
    for t in range(100):
        loss = m.forward_dec(ids, torch.zeros(2, 20, dtype=torch.long))[0].float().pow(2).mean() + ctl.orth_loss()
        opt.zero_grad(); loss.backward(); ctl.update(t); opt.step()
        if t >= 80:
            for mod, keep in zip(ctl.mods, ctl.final_mask):
                mod.lora_E.data.masked_fill_(~keep, 0.0)
    assert sum(ctl.ranks()) <= ctl.bT, (ctl.ranks(), ctl.bT)
    print(f"ok  AdaLoRA budget schedule {ctl.b0} -> {ctl.bT}, final ranks {ctl.ranks()}")

def test_decoding():
    s, _ = _build("LlamaLM")
    tok = FakeTok()
    P = [torch.randint(0, 290, (9,)), torch.randint(0, 290, (15,)), torch.randint(0, 290, (4,))]
    g = dict(GEN_DEFAULTS, max_new=20, min_new=5, no_repeat_ngram=3, batch_size=2)
    greedy, _ = generate(s, P, dict(g, method="greedy"), tok, "cpu")
    beam1, _ = generate(s, P, dict(g, method="beam", beams=1), tok, "cpu")
    assert greedy == beam1, (greedy, beam1)
    beam4, _ = generate(s, P, dict(g, method="beam", beams=4), tok, "cpu")
    samp, _ = generate(s, P, dict(g, method="sample"), tok, "cpu")
    for out in (greedy, beam4, samp):
        for o in out:
            assert len(o) <= 20 and not set(o) & set(tok.special_to_id.values())
            grams = [tuple(o[i:i + 3]) for i in range(len(o) - 2)]
            assert len(grams) == len(set(grams)), o
    lg = torch.zeros(1, V); lg[0, FakeTok.special_to_id["<|endoftext|>"]] = 10
    sp = {"eot": V - 5, "banned": torch.tensor([V - 4, V - 3, V - 2, V - 1])}
    assert process_logits(lg.clone(), torch.zeros(1, 2, dtype=torch.long), 2, dict(g), sp).argmax() != V - 5
    assert process_logits(lg.clone(), torch.zeros(1, 6, dtype=torch.long), 6, dict(g), sp).argmax() == V - 5
    sb, _ = _build("EncDecLM")
    Pb = [[torch.randint(0, 290, (20,)), torch.randint(0, 290, (7,))], [torch.randint(0, 290, (12,))]]
    gb, _ = generate(sb, Pb, dict(g, method="greedy"), tok, "cpu")
    bb, _ = generate(sb, Pb, dict(g, method="beam", beams=1), tok, "cpu")
    assert gb == bb
    generate(sb, Pb, dict(g, method="beam", beams=3), tok, "cpu")
    print("ok  decoding: beam(1) == greedy (A and B), no repeated 3-grams, specials never generated, min length")

def test_kd_loss():
    torch.manual_seed(0)
    s, t, lab = torch.randn(7, 11), torch.randn(7, 11), torch.randint(0, 11, (7,))
    l0, _ = seq_loss(s, lab)
    l1, p = seq_loss(s, lab, s.clone(), 0.5, 2.0)
    assert abs(p["kd"]) < 1e-5 and abs(float(l1) - 0.5 * float(l0)) < 1e-5
    l2, p2 = seq_loss(s, lab, t, 0.5, 2.0)
    assert p2["kd"] > 0
    print("ok  KD loss (zero for identical teacher, positive otherwise)")

def test_metrics():
    a = "The appeal is allowed and the conviction under Section 302 IPC is set aside."
    b = "Appeal allowed; conviction under section 302 of the IPC set aside by this Court."
    f = rouge_fallback(a, b)
    assert 0 < f["rouge1"][2] < 1 and rouge_fallback(a, a)["rougeL"][2] == 1.0
    try:
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False).score(b, a)
        assert abs(sc["rouge1"].fmeasure - f["rouge1"][2]) < 0.02 and abs(sc["rougeL"].fmeasure - f["rougeL"][2]) < 0.02
        r = rouge_scores([a], [b])
        assert r["rouge_impl"] == "rouge_score"
    except ImportError:
        print("    (rouge_score not installed: fallback only)")
    facts = legal_facts("Under Section 302 IPC and Article 21, the Indian Penal Code Act, 1860 applies; "
                        "decided on 12.03.1999 in Criminal Appeal No. 45 of 1998, fine of Rs. 5,000.")
    assert {"section 302", "article 21", "12 03 1999", "rs 5 000"} <= facts, facts
    assert redundancy("a b c a b c a b c")[0] > 0.5 and redundancy("one two three four")[0] == 0
    assert postprocess("This is a sentence. This is a sentence. Another one here.", {"dedup_sentences": True}) == \
        "This is a sentence. Another one here."
    print("ok  metrics (ROUGE fallback ~ rouge_score, legal facts, redundancy, sentence de-duplication)")

def test_submission():
    ids = ["doc_1", "doc_2", 3]
    d = tempfile.mkdtemp()
    zp = write_submission(ids, ["a summary", "b", "c"], d)
    import zipfile
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
        obj = json.loads(z.read("prediction.json"))
    assert obj[2] == {"id": 3, "summary": "c"}
    for bad in ([{"id": "doc_1", "summary": "x"}],
                [{"id": i, "summary": "x"} for i in ["doc_1", "doc_2", "doc_9"]],
                [{"id": i, "summary": ""} for i in ids],
                [{"id": i, "summary": "x", "extra": 1} for i in ids]):
        try:
            check_prediction(bad, ids); raise AssertionError(f"accepted {bad}")
        except AssertionError as e:
            if "accepted" in str(e):
                raise
    print("ok  submission (list of {id, summary}, exact ids, non-empty, zip root)")

def test_checkpoints(ckpt_dir):
    from p6summ.train import CKPT_FOLDERS
    found = 0
    for key, folder in CKPT_FOLDERS.items():
        p = os.path.join(ckpt_dir, folder, "model_final.pt")
        if not os.path.exists(p):
            continue
        found += 1
        m, cls, kw = load_phase1(p)
        m.eval()
        s = Summarizer(p).eval()
        s.bos_id = 31998
        torch.manual_seed(0)
        if cls == "EncDecLM":
            s.set_lengths(kw["dec_block"])
            src, dec = torch.randint(0, 31000, (1, 64)), torch.randint(0, 31000, (1, 16))
            with torch.no_grad():
                r, _ = m(src, torch.ones(1, 64, dtype=torch.bool), dec)
                g, _ = s.forward_encdec(src, torch.ones(1, 64, dtype=torch.bool), torch.zeros(1, dtype=torch.long),
                                        torch.zeros(1, dtype=torch.long), 1, 1, dec, torch.zeros(1, 16, dtype=torch.long))
            d = (r.reshape(-1, r.size(-1)) - g).abs().max().item()
        else:
            s.set_lengths(kw.get("block_size", 1024))
            ids = torch.randint(0, 31000, (1, 80))
            with torch.no_grad():
                r, _ = m(ids, torch.tensor([40])) if cls == "HybridLM" else m(ids)
                g, _ = s.forward_dec(ids, torch.zeros(1, 80, dtype=torch.long), torch.tensor([40]))
                d = (r.reshape(-1, r.size(-1)) - g).abs().max().item()
                st = s.prefill([ids[0, :40]])
                gl = [st["logits"][0]] + [s.step(st, ids[0, 40 + k:41 + k])[0] for k in range(5)]
                d = max(d, (torch.stack(gl) - r[0, 39:45]).abs().max().item()) if cls != "HybridLM" else d
        assert d < 2e-3, (folder, d)
        print(f"ok  checkpoint {folder} ({cls}, {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M): max |diff| {d:.2e}")
        del m, s
    if not found:
        print("    no Phase-1 checkpoints found under", ckpt_dir)

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt_dir", default="")
    a = ap.parse_args()
    test_sentences_and_context()
    test_loader()
    test_targets_and_collate()
    test_equivalence()
    test_peft()
    test_adalora_controller()
    test_decoding()
    test_kd_loss()
    test_metrics()
    test_submission()
    if a.ckpt_dir:
        test_checkpoints(a.ckpt_dir)
    print("ALL TESTS PASSED")

In [ ]:
for m in [m for m in list(sys.modules) if m.startswith("p6summ")]:
    del sys.modules[m]
from p6summ import data as D
from p6summ.train import run, resolve_ckpt, CKPT_FOLDERS, merge, set_by_path, DEFAULTS, load_data
from p6summ.run import (coverage_analysis, mbr_ensemble, length_sweep, generation_variants, bertscore_run, regenerate)
from p6summ.tokenizer import BPETokenizer
print("pipeline loaded")

## SETTINGS: the only cell you edit

In [ ]:
RUN_LIST = ["T1_A2_full", "R2_B_lora_fid", "R1_A2_lorma", "R3_C_pfeiffer", "R4_KD_D6_from_T1", "X1_D6_noKD",
            "X2_A2_qlora", "X4_B_houlsby_fid", "X3_A2_prefix", "X5_A2_lorma_salient", "X6_C_adalora", "X7_A2_lorma_dapt",
            "X8_B_lora_768"]
OVERRIDES   = {}
SMOKE_TEST  = False
RUN_TESTS   = True
RUN_EDA     = True
PARALLEL    = True
MAX_MINUTES = 55
SKIP_FINISHED = True
TAG         = ""
USE_BERTSCORE = True
ENSEMBLE    = "auto"
ANALYSIS_RUN = "auto"
ANALYSIS_DOCS = 100
REGENERATE  = {}

A2 = {"ckpt": "@A2"}
LORMA = {"name": "lorma", "variant": "plus", "r": 16, "alpha": 32, "targets": ["attn", "ffn"]}
D6 = {"backbone": {"ckpt": "@D", "keep_layers": 6}, "method": {"name": "full"},
      "context": {"src_len": 1536},
      "train": {"lr_backbone": 1e-4}}
B_FID = {"src_len": 3072}
EXPERIMENTS = {
    'T1_A2_full':          {'backbone': dict(A2), 'method': {'name': 'full'}, 'train': {'lr_backbone': 5e-05}},
    'R1_A2_lorma':         {'backbone': dict(A2), 'method': dict(LORMA), 'train': {'lr_new': 5e-04}},
    'R2_B_lora_fid':       {'backbone': {'ckpt': '@B'}, 'method': {'name': 'lora', 'r': 16, 'alpha': 32},
                            'context': dict(B_FID), 'train': {'lr_new': 5e-04}},
    'R3_C_pfeiffer':       {'backbone': {'ckpt': '@C'}, 'method': {'name': 'pfeiffer', 'bottleneck': 128}, 'train': {'lr_new': 1e-03}},
    'R4_KD_D6_from_T1':    {**D6, 'kd': {'teacher_run': 'T1_A2_full', 'kd_alpha': 0.5, 'kd_T': 2.0}},
    'X1_D6_noKD':          dict(D6),
    'X2_A2_qlora':         {'backbone': dict(A2), 'method': {'name': 'qlora', 'r': 16, 'alpha': 32}, 'train': {'lr_new': 5e-04}},
    'X3_A2_prefix':        {'backbone': dict(A2), 'method': {'name': 'prefix', 'n_prefix': 32}, 'train': {'lr_new': 3e-03}},
    'X4_B_houlsby_fid':    {'backbone': {'ckpt': '@B'}, 'method': {'name': 'houlsby', 'bottleneck': 64},
                            'context': dict(B_FID), 'train': {'lr_new': 1e-03}},
    'X5_A2_lorma_salient': {'backbone': dict(A2), 'method': dict(LORMA), 'context': {'strategy': 'salient', 'lead': 256},
                            'train': {'lr_new': 5e-04}},
    'X6_C_adalora':        {'backbone': {'ckpt': '@C'}, 'method': {'name': 'adalora', 'r': 12, 'alpha': 24},
                            'adalora': {'target_r': 4}, 'train': {'lr_new': 1e-03}},
    'X7_A2_lorma_dapt':    {'backbone': dict(A2), 'method': dict(LORMA), 'dapt': {'steps': 300, 'len': 1024},
                            'train': {'lr_new': 5e-04}},
    'X8_B_lora_768':       {'backbone': {'ckpt': '@B'}, 'method': {'name': 'lora', 'r': 16, 'alpha': 32},
                            'context': {'src_len': 768}, 'train': {'lr_new': 5e-04}},
    'Y1_D12_lora_long':    {'backbone': {'ckpt': '@D'}, 'method': {'name': 'lora', 'r': 16, 'alpha': 32},
                            'context': {'src_len': 3072}, 'train': {'lr_new': 5e-04}},
    'Y2_A2_random_full':   {'backbone': {'ckpt': '@A2', 'random_init': True}, 'method': {'name': 'full'},
                            'train': {'lr_backbone': 3e-04}},
    'Y4_C_lorma':          {'backbone': {'ckpt': '@C'}, 'method': dict(LORMA), 'train': {'lr_new': 5e-04}},
    'Y5_A2_prompt':        {'backbone': dict(A2), 'method': {'name': 'prompt', 'n_prompt': 32}, 'train': {'lr_new': 3e-03}},
}
OUT_DIR = f"{WORK}/runs"
CACHE_DIR = f"{WORK}/p6cache"

## Data, checkpoints and the one-time tokenisation

In [ ]:
DATA_DIR = D.find_data_dir([os.environ.get("P6_DATA_DIR", ""), f"{WORK}/p6data/Phase_6_Summ"])
if DATA_DIR is None:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("could not read Kaggle secret HF_TOKEN:", e)
    DATA_DIR = D.download(f"{WORK}/p6data")
print("data:", DATA_DIR)
CKPT_ROOT = os.environ.get("P6_CKPT_ROOT", "")
FOUND = {}
for key, folder in CKPT_FOLDERS.items():
    try:
        FOUND[key] = resolve_ckpt("@" + key, [CKPT_ROOT] if CKPT_ROOT else [])
        print(f"  @{key:3s} OK      {FOUND[key]}")
    except FileNotFoundError:
        print(f"  @{key:3s} MISSING {folder}/model_final.pt  -> runs using it will be skipped")
assert FOUND, "no Phase-1 checkpoint found: attach the cs781-phase1-ckpts dataset"
TOK_PATH = os.path.join(os.path.dirname(FOUND.get("A2") or next(iter(FOUND.values()))), "tokenizer.json")
TOK = BPETokenizer.load(TOK_PATH)
MAX_DOCS = 24 if SMOKE_TEST else None
BASE = {"data_dir": str(DATA_DIR), "out_dir": OUT_DIR, "cache_dir": CACHE_DIR, "max_docs": MAX_DOCS,
        "backbone": {"ckpt": FOUND.get("A2") or next(iter(FOUND.values()))}}
import time; t0 = time.time()
DATA = load_data(merge(DEFAULTS, BASE))
print(f"train {len(DATA['train']['ids'])} | dev {len(DATA['dev']['ids'])} (held out from train.json) | "
      f"test {len(DATA['test']['ids'])} documents | ready in {time.time() - t0:.0f}s")

In [ ]:
if RUN_TESTS:
    ck_dirs = sorted({os.path.dirname(os.path.dirname(p)) for p in FOUND.values()})
    for cd in ck_dirs or [""]:
        r = subprocess.run([sys.executable, "tests/test_pipeline.py"] + (["--ckpt_dir", cd] if cd else []), cwd=WORK,
                           capture_output=True, text=True)
        print(r.stdout[-3000:], r.stderr[-3000:])
        assert r.returncode == 0, "a correctness check failed - fix it before training"

## Data analysis

In [ ]:
if RUN_EDA:
    import numpy as np, matplotlib.pyplot as plt, pandas as pd
    D.describe_raw(DATA_DIR)
    ST = D.stats(DATA)
    print(json.dumps(ST, indent=1))
    POS = D.summary_source_positions(DATA, "train", max_docs=500)
    print("relative position (10 bins, start -> end of the document) of the sentence each summary sentence is taken from:", POS)
    tr = DATA["train"]
    dl = np.array([len(t) for t in tr["tokens"]]); sl = np.array([len(t) for t in tr["sum_tokens"]])
    fig, ax = plt.subplots(1, 4, figsize=(21, 3.6))
    ax[0].hist(np.log10(np.maximum(dl, 1)), bins=60)
    for L in (768, 1400, 3072):
        ax[0].axvline(np.log10(L), color="k", ls=":", lw=1)
    ax[0].set_xlabel("log10(document tokens)  (dotted: 768 / 1400 / 3072)"); ax[0].set_title("document length (train)")
    ax[1].hist(sl, bins=60); ax[1].axvline(640, color="k", ls=":", lw=1)
    ax[1].set_xlabel("summary tokens (dotted: 640)"); ax[1].set_title("reference summary length")
    ax[2].scatter(dl, sl, s=3, alpha=0.3); ax[2].set_xscale("log"); ax[2].set_yscale("log")
    ax[2].set_xlabel("document tokens"); ax[2].set_ylabel("summary tokens"); ax[2].set_title("summary vs document length")
    ax[3].bar(np.arange(10) / 10 + 0.05, POS, width=0.09); ax[3].set_xlabel("relative position in the document")
    ax[3].set_title("where summary sentences come from"); plt.tight_layout(); plt.show()
    COV = coverage_analysis(DATA, TOK, budgets=(512, 768, 1400, 1536, 3072), max_docs=200)
    print(f"reference content words present in the WHOLE document: {COV['whole_document']:.3f} (upper bound)")
    display(pd.DataFrame(COV["rows"]))
    json.dump({"stats": ST, "source_positions": POS, "coverage": COV}, open(f"{WORK}/data_analysis.json", "w"), indent=1)

## Build the job list

In [ ]:
def run_name(n):
    return n + TAG + ("_smoke" if SMOKE_TEST else "")

def finished(name):
    return os.path.exists(os.path.join(OUT_DIR, name, "summary.json"))

JOBS = []
for name in RUN_LIST:
    cfg = json.loads(json.dumps(EXPERIMENTS[name]))
    cfg.update(run_name=run_name(name), data_dir=str(DATA_DIR), out_dir=OUT_DIR, ckpt_root=CKPT_ROOT,
               cache_dir=CACHE_DIR, max_docs=MAX_DOCS)
    key = cfg["backbone"]["ckpt"].lstrip("@")
    if key not in FOUND:
        print(f"SKIP {name}: checkpoint @{key} not attached"); continue
    cfg["backbone"]["ckpt"] = FOUND[key]
    teacher = None
    if cfg.get("kd", {}).get("teacher_run"):
        teacher = run_name(os.path.basename(cfg["kd"]["teacher_run"]))
        cfg["kd"]["teacher_run"] = os.path.join(OUT_DIR, teacher)
        if teacher not in [run_name(n) for n in RUN_LIST] and not finished(teacher):
            print(f"SKIP {name}: teacher {teacher} is neither in RUN_LIST nor finished earlier"); continue
    cfg.setdefault("train", {})["max_minutes"] = MAX_MINUTES
    for k, v in OVERRIDES.items():
        set_by_path(cfg, k, json.dumps(v))
    if SMOKE_TEST:
        cfg["backbone"]["keep_layers"] = 1 if not cfg["backbone"].get("keep_layers") else 2
        cfg["train"].update(epochs=1, docs_per_epoch=8, gen_eval_docs=2, max_minutes=5)
        cfg.setdefault("context", {})["src_len"] = min(cfg.get("context", {}).get("src_len", 1400), 512)
        cfg["target"] = {"tgt_cap": 64}
        cfg["eval"] = {"gen_docs": 8}
        if cfg.get("dapt"):
            cfg["dapt"]["steps"] = 4
    if SKIP_FINISHED and finished(cfg["run_name"]):
        print(f"  {cfg['run_name']:28s} already finished -> skipped (SKIP_FINISHED)"); continue
    JOBS.append((cfg["run_name"], cfg, teacher))
for n, c, t in JOBS:
    cx = merge(DEFAULTS["context"], c.get("context", {}))
    print(f"  {n:28s} model {os.path.basename(os.path.dirname(c['backbone']['ckpt'])):22s} "
          f"layers {c['backbone'].get('keep_layers') or 'all':4} method {c['method']['name']:9s} "
          f"context {cx['src_len']} {cx['strategy']:8s}" + (f"  (teacher {t})" if t else ""))

## Run (parallel on 2 GPUs, otherwise sequential)

In [ ]:
import time, traceback
os.makedirs(f"{OUT_DIR}/_jobs", exist_ok=True); os.makedirs(f"{OUT_DIR}/_logs", exist_ok=True)
KEEP = ("[epoch]", "[submit]", "[kd]", "[train]", "[model]", "[dev]", "[dapt]", "[adalora]", "[data]", "early stop",
        "Error", "Traceback")

def run_parallel(jobs, gpus):
    pending, running, done, failed = list(jobs), {}, set(), set()
    t0 = time.time()
    while pending or running:
        for g in gpus:
            if g in running:
                continue
            for j in list(pending):
                name, cfg, teacher = j
                if teacher and teacher in failed:
                    print(f"SKIP {name}: teacher {teacher} failed"); pending.remove(j); failed.add(name); continue
                if teacher and not finished(teacher):
                    continue
                path = f"{OUT_DIR}/_jobs/{name}.json"
                json.dump(cfg, open(path, "w"), indent=1)
                log = open(f"{OUT_DIR}/_logs/{name}.log", "w")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED="1")
                proc = subprocess.Popen([sys.executable, "-u", "-m", "p6summ.run", "--config", path],
                                        cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT)
                running[g] = (name, proc, log, 0)
                pending.remove(j)
                print(f"[{time.time()-t0:6.0f}s] START {name} on GPU {g}")
                break
        time.sleep(15)
        for g, (name, proc, log, pos) in list(running.items()):
            txt = open(f"{OUT_DIR}/_logs/{name}.log").read()
            for line in txt[pos:].splitlines():
                if any(k in line for k in KEEP):
                    print(f"  [{name} @GPU{g}] {line}")
            running[g] = (name, proc, log, len(txt))
            if proc.poll() is not None:
                log.close(); del running[g]
                if proc.returncode == 0 and finished(name):
                    done.add(name); print(f"[{time.time()-t0:6.0f}s] DONE  {name}")
                else:
                    failed.add(name); print(f"[{time.time()-t0:6.0f}s] FAILED {name} (exit {proc.returncode}); log tail:")
                    print("\n".join(txt.splitlines()[-25:]))
        if pending and not running and all(t and not finished(t) for _, _, t in pending):
            for n, _, t in pending:
                print(f"SKIP {n}: teacher {t} never finished")
            break
    return done, failed

if PARALLEL and N_GPU >= 2:
    DONE, FAILED = run_parallel(JOBS, list(range(N_GPU)))
else:
    DONE, FAILED = set(), set()
    for name, cfg, teacher in JOBS:
        if teacher and not finished(teacher):
            print(f"SKIP {name}: teacher {teacher} not finished"); continue
        print(f"\n##################### {name} #####################")
        try:
            summary, model = run(cfg, data=DATA); del model; torch.cuda.empty_cache(); DONE.add(name)
        except Exception:
            traceback.print_exc(); FAILED.add(name)
            print(f">>> {name} FAILED; continuing")

os.makedirs(f"{WORK}/submissions", exist_ok=True)
for p in sorted(glob.glob(f"{OUT_DIR}/*/summary.json")):
    s = json.load(open(p))
    if s["run"].endswith("_smoke") or not s.get("submission") or "params_total" not in s:
        continue
    shutil.copy(s["submission"], f"{WORK}/submissions/{s['run']}.zip")
print("done:", sorted(DONE), "| failed:", sorted(FAILED))
print("submissions:", sorted(os.listdir(f"{WORK}/submissions")))

## Results table (also saved as `/kaggle/working/RESULTS.csv`)

In [ ]:
import pandas as pd, numpy as np
from p6summ.metrics import free_bertscore
BERT = {"lang": "en", "batch_size": 16, "device": "cuda" if N_GPU else "cpu"} if USE_BERTSCORE else None
rows = []
for p in sorted(glob.glob(f"{OUT_DIR}/*/summary.json")):
    s = json.load(open(p))
    if "params_total" not in s or SMOKE_TEST != s["run"].endswith("_smoke"):
        continue
    bs = None
    if BERT:
        try:
            bs = bertscore_run(os.path.dirname(p), BERT)
        except Exception as e:
            print("BERTScore unavailable:", repr(e)[:200]); BERT = None
    d, cx, r = s["dev"], s["context"], s["resolved"]
    rows.append({"run": s["run"], "arch": s["arch"], "method": s["method"]["name"] + ("+KD" if s.get("kd_teacher") else "")
                 + ("+DAPT" if s.get("dapt") else ""), "layers": s["layers"],
                 "context": f"{r['src_len']} {cx['strategy']}", "params_M": round(s["params_total"] / 1e6, 1),
                 "trainable_M": round(s["params_trainable"] / 1e6, 3), "trainable_%": s["trainable_pct"],
                 "train_min": s["train_minutes"], "best_epoch": s["best_epoch"],
                 "peak_mem_GB": round(s["peak_train_mem_gb"] or 0, 2), "gen_docs_s": s["gen_docs_per_sec"],
                 "dev_loss_0": s["dev_loss_before"], "dev_loss": round(s["best_dev_loss"], 4),
                 "dev_BERTScore_F1": round(bs["bertscore_f"], 4) if bs else np.nan,
                 "dev_R1": round(d["rouge1_f"], 4), "dev_R2": round(d["rouge2_f"], 4), "dev_RL": round(d["rougeL_f"], 4),
                 "words": round(d["words"]), "ref_words": round(d["ref_words"]), "rep3": round(d["rep_3gram"], 4),
                 "novel3": round(d["novel_3gram"], 4), "fact_P": round(d["fact_precision"], 4),
                 "fact_R": round(d["fact_recall"], 4), "test_words": s.get("test_words")})
RES = pd.DataFrame(rows)
if rows:
    key = "dev_BERTScore_F1" if RES["dev_BERTScore_F1"].notna().any() else "dev_R1"
    RES = RES.sort_values(key, ascending=False).reset_index(drop=True)
    RES.to_csv(f"{WORK}/RESULTS.csv", index=False)
    pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250)
    display(RES)
else:
    print("no finished runs")

## Training curves (dev loss per epoch; epoch 0 = the Phase-1 model before fine-tuning)

In [ ]:
import matplotlib.pyplot as plt
if rows:
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
    for r in RES["run"]:
        h = json.load(open(f"{OUT_DIR}/{r}/history.json"))
        ax[0].plot([x["epoch"] for x in h], [x["dev_loss"] for x in h], marker="o", label=r)
        q = [x for x in h if "quick_rougeL" in x]
        if q:
            ax[1].plot([x["epoch"] for x in q], [x["quick_rougeL"] for x in q], marker="o", label=r)
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("dev loss (summary tokens)"); ax[0].set_ylim(top=min(ax[0].get_ylim()[1], 8))
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("ROUGE-L (16 dev docs, greedy, 256 tokens)")
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## Ensemble → one more submission

In [ ]:
if ENSEMBLE == "auto":
    members = [r for r in RES["run"] if not r.startswith("ENS")][:4] if rows else []
else:
    members = [run_name(n) for n in ENSEMBLE]
members = [m for m in members if finished(m)]
ENS_NAME = f"ENS_MBR{TAG}" + ("_smoke" if SMOKE_TEST else "")
if len(members) >= 2:
    es = mbr_ensemble([f"{OUT_DIR}/{r}" for r in members], f"{OUT_DIR}/{ENS_NAME}", DATA)
    eb = bertscore_run(f"{OUT_DIR}/{ENS_NAME}", BERT) if BERT else None
    print("MBR over", members, "| dev ROUGE-1", round(es["dev"]["rouge1_f"], 4), "| ROUGE-L", round(es["dev"]["rougeL_f"], 4),
          "| BERTScore F1", round(eb["bertscore_f"], 4) if eb else "-", "| picks", es["dev_picks"])
    if not SMOKE_TEST:
        shutil.copy(es["submission"], f"{WORK}/submissions/{ENS_NAME}.zip")
else:
    print("need >= 2 finished runs for the ensemble")

## How does summary length affect BERTScore and ROUGE?

In [ ]:
AN = None
if rows:
    AN = RES["run"].iloc[0] if ANALYSIS_RUN == "auto" else run_name(ANALYSIS_RUN)
    print("analysis run:", AN)
    LS = pd.DataFrame(length_sweep(f"{OUT_DIR}/{AN}", DATA, bert=BERT))
    display(LS)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    x = LS["mean_words"]
    for c in [c for c in ("bertscore_f", "rouge1_f", "rouge1_r", "rougeL_f") if c in LS]:
        ax[0].plot(x, LS[c], marker="o", label=c)
    ax[0].set_xlabel("mean summary words"); ax[0].set_title(f"{AN}: truncation sweep"); ax[0].legend()
    for c in [c for c in ("dev_BERTScore_F1", "dev_R1", "dev_RL") if RES[c].notna().any()]:
        ax[1].scatter(RES["words"], RES[c], label=c)
    for _, r in RES.iterrows():
        ax[1].annotate(r["run"][:10], (r["words"], r["dev_R1"]), fontsize=7)
    ax[1].set_xlabel("mean summary words (run)"); ax[1].set_title("length vs score across runs"); ax[1].legend()
    plt.tight_layout(); plt.show()
    json.dump(LS.to_dict("records"), open(f"{OUT_DIR}/{AN}/length_sweep.json", "w"), indent=1)

## Decoding strategies and redundancy (analysis run, `ANALYSIS_DOCS` dev documents)

In [ ]:
if AN:
    spec = json.load(open(f"{OUT_DIR}/{AN}/config.json"))["resolved"]
    DEC = {"greedy, no-repeat-4 (default)": {},
           "beam 4": {"gen": {"method": "beam", "beams": 4}},
           "sampling T=0.7 top-p 0.9": {"gen": {"method": "sample"}},
           "greedy, no n-gram blocking": {"gen": {"no_repeat_ngram": 0}},
           "greedy, no blocking + rep. penalty 1.2": {"gen": {"no_repeat_ngram": 0, "rep_penalty": 1.2}},
           "greedy, max length / 2": {"gen": {"max_new": max(32, spec["max_new"] // 2), "min_new": min(spec["min_new"], spec["max_new"] // 4)}}}
    DV = pd.DataFrame(generation_variants(f"{OUT_DIR}/{AN}", DATA, DEC, ANALYSIS_DOCS, "cuda" if N_GPU else "cpu", BERT))
    display(DV)
    json.dump(DV.to_dict("records"), open(f"{OUT_DIR}/{AN}/decoding_variants.json", "w"), indent=1)

## Which parts of a long judgment matter? (context views at inference)

In [ ]:
if AN:
    CV = {f"{s}{' (trained view)' if s == json.load(open(f'{OUT_DIR}/{AN}/config.json'))['context']['strategy'] else ''}":
          {"context": {"strategy": s}} for s in ("head", "tail", "headtail", "uniform", "salient")}
    CVR = pd.DataFrame(generation_variants(f"{OUT_DIR}/{AN}", DATA, CV, ANALYSIS_DOCS, "cuda" if N_GPU else "cpu", BERT))
    display(CVR)
    json.dump(CVR.to_dict("records"), open(f"{OUT_DIR}/{AN}/context_views.json", "w"), indent=1)
    pair = [run_name(n) for n in ("R1_A2_lorma", "X5_A2_lorma_salient", "X7_A2_lorma_dapt", "R2_B_lora_fid", "X8_B_lora_768")]
    display(RES[RES["run"].isin(pair)][["run", "method", "context", "dev_loss", "dev_BERTScore_F1", "dev_R1", "dev_RL", "fact_R"]])

## Coverage, factuality and redundancy of every run; quality by reference length

In [ ]:
if rows:
    display(RES[["run", "method", "words", "ref_words", "dev_R1", "rep3", "novel3", "fact_P", "fact_R"]])
    from p6summ.metrics import length_bucket_report
    best = RES["run"].iloc[0]
    g = json.load(open(f"{OUT_DIR}/{best}/dev_generations.json"))
    sc = "bertscore_f" if "bertscore_f" in g[0] else "rouge1_f"
    print(f"{best}: {sc} by reference length")
    display(pd.DataFrame(length_bucket_report([x[sc] for x in g], [len(x["reference"].split()) for x in g])))

## Knowledge distillation and efficiency: model size, memory and inference cost

In [ ]:
if rows:
    key = "dev_BERTScore_F1" if RES["dev_BERTScore_F1"].notna().any() else "dev_R1"
    E = RES[["run", "arch", "method", "layers", "params_M", "trainable_M", "trainable_%", "peak_mem_GB", "train_min",
             "gen_docs_s", key]].copy()
    E["vs_best"] = (E[key] - E[key].max()).round(4)
    display(E.sort_values("params_M"))
    kd = RES[RES["run"].isin([run_name(n) for n in ("T1_A2_full", "R4_KD_D6_from_T1", "X1_D6_noKD")])]
    display(kd[["run", "params_M", "peak_mem_GB", "gen_docs_s", "dev_loss", "dev_BERTScore_F1", "dev_R1", "dev_RL"]])

## Examples (first dev document: reference vs the three best runs, first 120 words)

In [ ]:
if rows:
    import textwrap
    g0 = {r: json.load(open(f"{OUT_DIR}/{r}/dev_generations.json"))[0] for r in RES["run"][:3]}
    first = next(iter(g0.values()))
    print("REFERENCE:\n" + textwrap.fill(" ".join(first["reference"].split()[:120]), 140))
    for r, x in g0.items():
        print(f"\n{r}:\n" + textwrap.fill(" ".join(x["generated"].split()[:120]), 140))

## Optional: extra submissions from finished runs with other decoding (`REGENERATE` in SETTINGS)

In [ ]:
for rn, g in REGENERATE.items():
    src = f"{OUT_DIR}/{run_name(rn)}"
    if not finished(run_name(rn)):
        print("skip", rn, "(not finished)"); continue
    out = f"{src}_{g.get('method', 'greedy')}{g.get('beams', '') if g.get('method') == 'beam' else ''}"
    s = regenerate(src, DATA, out, gen=g, device="cuda" if N_GPU else "cpu")
    b = bertscore_run(out, BERT) if BERT else None
    print(os.path.basename(out), "| dev ROUGE-1", round(s["dev"]["rouge1_f"], 4), "| BERTScore F1", round(b["bertscore_f"], 4) if b else "-")
    if not SMOKE_TEST:
        shutil.copy(s["submission"], f"{WORK}/submissions/{os.path.basename(out)}.zip")
free_bertscore()

## Download / submit